# This document contains a few robot PLANT simulation **classes** and verifies their behaviour is a simple **open loop** simulation
* Each Robot Class implements a modular physics engine of the plant that can be executed at each time step $\Delta T_{model}$, and contains its own State Vector class. 
* Each Robot Class contains `step`, `get_Hmatrix` and `_init_` Functions that passes the same variables back for drop in place substitution.  
* All numerical paramaters are not defined inside classes and furthermore all classes use the same paramater dictionary `Simulation_Paramaters`.
* All robot classes use 4 inputs (1 per wheel)

1. `OmniRobotSimulationExplicit` used `OmniRobotStateExplicit`:
    * Preserves and moves integrated memory states: armature currents, wheel speeds, LuGre bristle deflections, local/global velocities, and position coordinates
    * Evaluates LuGre bristle deflection states $\mathbf{z}_{k+1}$ using an **unconditionally stable exponential discrete update**, computing continuous traction forces and smoothed rolling drag.
    * Integrates Newton-Euler rigid chassis motion and maps body velocities into global coordinates.
2. `OmniRobotSimulationODESolve` uses `OmniRobotStateODESolve`:
    * 
    * 




```text
==========================================================================
                     SPATIAL REFERENCE FRAME LOGIC
==========================================================================

   GLOBAL FRAME (Fixed Room)               LOCAL FRAME (Robot Centric)
   
          ^+Y_global                              ^+Y_robot (Forward)
          |                                       |
          |                                       |
          +-------> +X_global                     +-------> +X_robot (Right/Strafing)
         /                                       /
        v +Z_global (Up)                        v +Z_robot (Up)

   • Orientation Heading (θ): Angle measured counter-clockwise from +Y_global to +Y_robot.
   • Wheel 1: Front Right (Quadrant I)   at angle +alpha_f relative to +X_robot
   • Wheel 2: Front Left  (Quadrant II)  at angle +alpha_f relative to -X_robot
   • Wheel 3: Rear Left   (Quadrant III) at angle +alpha_b relative to -X_robot
   • Wheel 4: Rear Right  (Quadrant IV)  at angle +alpha_b relative to +X_robot
==========================================================================
```


In [2]:
class OmniRobotObservableState:
    """
    Data container of data that the controller observes at every time step.
    """
    def __init__(self):
        # Memory Integration States (Differential Cores)
        self.I = np.zeros(4)            # Armature currents [i1, i2, i3, i4] (A)
        self.omega = np.zeros(4)        # Wheel angular velocities [w1, w2, w3, w4] (rad/s)
        self.pos_global = np.zeros(3)   # Global coordinates [x, y, theta] (m, rad)
        self.V_actual = np.zeros(4)     # Clipped applied terminal voltages [V1, V2, V3, V4] (V)
        


### `OmniRobotSimulationExplicit`  Specification: Discrete-Time Open-Loop Asymmetric 4-Omni Robot Model

This cell specifies the comprehensive mathematical framework for the 4-wheel omni-directional mobile robot simulation. The architecture accounts for **asymmetric wheel layouts**, **dynamic multi-axis weight transfer (pitch and roll)**, **hardware-calibrated motor parameters**, and **LuGre bristle floor friction boundaries** evaluated via **implicit exponential discrete-time updates** at a fixed sampling interval $\Delta T = 1\text{ ms}$ ($1\text{ kHz}$ loop rate).

---
<details>
<summary><b> Open this for the full specification of the simulation equations</b></summary>


### 1. Parameters & Variable Definitions

#### 🛠️ Hardware & Structural Parameters
* $M_r$: Total mass of the robot chassis (kg)
* $J_{zz}$: Rotational mass moment of inertia around vertical Z-axis (kg·m²)
* $R_b$: Geometric base radius from Center of Mass (CoM) to wheel contact points (m)
* $r$: Operating radius of omni-wheel tires (m)
* $\alpha_f$: Front wheel mounting angle relative to local $+x_r$ axis (rad)
* $\alpha_b$: Rear wheel mounting angle relative to local $+x_r$ axis (rad)
* $h$: Height of Center of Mass (CoM) above ground plane (m)

#### ⚡ Motor Specifications & Derived Parameters
* **Motor Drive**: Brushed DC Motor with integrated gear reduction and H-bridge drive electronics.
* $R$: Armature winding resistance ($\Omega$)
* $L_m$: Armature winding inductance (H)
* $K_t, K_b$: Electromechanical torque constant (N·m/A) and back-EMF constant (V/(rad/s))
* $J_w$: Reflected rotational inertia of motor rotor, gearhead, and wheel assembly (kg·m²)
* $V_{\max}$: Terminal supply voltage saturation limit (V)
* $I_{\max}$: Motor driver over-current protection ceiling (A)
* $C_{rr}$: Floor-tire rolling resistance coefficient
* $\mu_s, \mu_k$: Static and kinetic floor-tire friction coefficients
* $\delta$: LuGre presliding bristle deflection limit (m)
* $\zeta$: LuGre bristle damping ratio fraction against rim inertia
* $v_\sigma$: Slip velocity threshold where bristle damping fades out (m/s)
* $v_c$: Creep regularization velocity preventing division by zero at rest (m/s)
* $\alpha_d$: Stribeck exponential decay rate from static to kinetic friction (s/m)
* $\varepsilon$: Hyperbolic tangent smoothing speed for rolling resistance drag
* $v_{s,\text{limit}}$: Wheel sliding velocity threshold limit (delineates pure rolling / presliding from sliding) (rad/s)

---

### 2. Signal Tracker Definitions

At each sample time step $k$, the discrete simulation cycle processes the following variables:

#### System Inputs
* $\mathbf{V}_k = [V_{1,k}, V_{2,k}, V_{3,k}, V_{4,k}]^\top$: Armature terminal voltage commands (Volts).

#### Tracked Memory Integration States
* $\mathbf{I}_k = [i_{1,k}, i_{2,k}, i_{3,k}, i_{4,k}]^\top$: Motor armature currents (Amperes).
* $\mathbf{\Omega}_k = [\omega_{1,k}, \omega_{2,k}, \omega_{3,k}, \omega_{4,k}]^\top$: Wheel angular velocities (rad/s).
* $\mathbf{z}_k = [z_{1,k}, z_{2,k}, z_{3,k}, z_{4,k}]^\top$: LuGre normalized bristle deflections.
* $\mathbf{\dot{X}}_{r,k} = [\dot{x}_{r,k}, \dot{y}_{r,k}, \dot{\theta}_{k}]^\top$: Local chassis velocities (m/s Right, m/s Forward, rad/s Yaw).
* $\mathbf{\ddot{X}}_{r,k-1} = [a_{x,r,k-1}, a_{y,r,k-1}, \alpha_{r,k-1}]^\top$: Historical chassis accelerations from step $k-1$ (m/s², m/s², rad/s²).
* $\mathbf{\dot{X}}_{g,k} = [\dot{x}_{g,k}, \dot{y}_{g,k}, \dot{\theta}_{g,k}]^\top$: Global room frame velocities (m/s, m/s, rad/s).
* $\mathbf{X}_{g,k} = [x_k, y_k, \theta_k]^\top$: Global room frame navigation coordinates (m, m, rad).

---

### 3. Kinematic Mapping Matrix $\mathbf{H}$ (Body Frame: $+x_r$ Right, $+y_r$ Forward)

$$\mathbf{v}_{w,k} = \mathbf{H} \mathbf{\dot{X}}_{r,k}, \qquad \mathbf{H} = \begin{bmatrix} -\sin\alpha_f & \cos\alpha_f & R_b \\ -\sin\alpha_f & -\cos\alpha_f & R_b \\ \sin\alpha_b & -\cos\alpha_b & R_b \\ \sin\alpha_b & \cos\alpha_b & R_b \end{bmatrix}$$

---

### 4. Dynamic Normal Load Redistribution

With $+x_r$ Right (strafing) and $+y_r$ Forward (longitudinal):
$$y_f = R_b \sin\alpha_f, \qquad y_b = R_b \sin\alpha_b, \qquad L = y_f + y_b$$
$$x_f = R_b \cos\alpha_f, \qquad x_b = R_b \cos\alpha_b, \qquad \Sigma X^2 = x_f^2 + x_b^2$$

#### Static Normal Force Distribution at Rest
$$F_{n,\text{front, static}} = \frac{M_r g y_b}{2 L}, \qquad F_{n,\text{rear, static}} = \frac{M_r g y_f}{2 L}$$

#### Dynamic Weight Shifts (Pitch due to $a_{y,\text{past}}$, Roll due to $a_{x,\text{past}}$)
$$\Delta F_{n,y} = \frac{M_r a_{y,\text{past}} h}{2 L}, \qquad \Delta F_{n,x,\text{front}} = \frac{x_f}{\Sigma X^2} \frac{M_r a_{x,\text{past}} h y_b}{L}, \qquad \Delta F_{n,x,\text{rear}} = \frac{x_b}{\Sigma X^2} \frac{M_r a_{x,\text{past}} h y_f}{L}$$

$$\mathbf{F}_{n,k} = \begin{bmatrix} F_{n,\text{front, static}} - \Delta F_{n,y} - \Delta F_{n,x,\text{front}} + w_{F,1} \\ F_{n,\text{front, static}} - \Delta F_{n,y} + \Delta F_{n,x,\text{front}} + w_{F,2} \\ F_{n,\text{rear, static}} + \Delta F_{n,y} + \Delta F_{n,x,\text{rear}} + w_{F,3} \\ F_{n,\text{rear, static}} + \Delta F_{n,y} - \Delta F_{n,x,\text{rear}} + w_{F,4} \end{bmatrix}$$

---

### 5. LuGre Bristle Friction & Discrete Integration

1. **Contact Slip Velocity along Rolling Axis:**
   $$v_{s,i,k} = (\mathbf{H} \mathbf{\dot{X}}_{r,k})_i - r \omega_{i,k}$$

2. **Smoothed Rolling Resistance Drag:**
   $$T_{\text{roll},i,k} = C_{rr} F_{n,i,k} r \tanh\left( \frac{\omega_{i,k}}{\varepsilon} \right)$$

3. **Stribeck Friction Curve & Local Characteristic Time Constant:**
   $$\mu_{d,i,k} = \mu_k + (\mu_s - \mu_k) e^{-\alpha_d |v_{s,i,k}|}$$
   $$z_{\text{ss},i,k} = \frac{v_{s,i,k} \mu_{d,i,k}}{\mu_s \sqrt{v_{s,i,k}^2 + v_c^2}}, \qquad \tau_{i,k} = \frac{\delta \mu_{d,i,k}}{\mu_s \sqrt{v_{s,i,k}^2 + v_c^2}}$$

4. **Implicit Exponential Discrete Bristle Update ($\Delta T = 1\text{ ms}$):**
   $$z_{i,k+1} = z_{i,k} e^{-\Delta T / \tau_{i,k}} + z_{\text{ss},i,k} \left( 1 - e^{-\Delta T / \tau_{i,k}} \right), \qquad \dot{z}_{i,k} = \frac{z_{i,k+1} - z_{i,k}}{\Delta T}$$

5. **Net Tractive Floor Force & Unified Wheel Speed Update:**
   $$\sigma_{1,i,k} = 2 \zeta \sqrt{ \frac{\mu_s F_{n,i,k}}{\delta} \frac{J_w}{r^2} } e^{-(v_{s,i,k} / v_\sigma)^2}$$
   $$F_{i,k} = -\left( \mu_s F_{n,i,k} z_{i,k+1} + \sigma_{1,i,k} \delta \dot{z}_{i,k} + \sigma_2 v_{s,i,k} \right)$$
   $$\omega_{i,k+1} = \omega_{i,k} + \frac{\Delta T}{J_w} \left( T_{m,i,k} - T_{\text{roll},i,k} - r F_{i,k} \right)$$

</details>


---
<details>
<summary><b> Open this for simulation flowchart </b></summary>


```text
==================================================
        SIMULATION STATE DATA (STEP K)
==================================================
 [Inputs] -> Voltages Vector V_1..4
 [Memory] -> Current i_k, Wheel Speeds omega_k
 [Memory] -> LuGre Bristle Deflections z_k
 [Memory] -> Local Chassis Speeds & Past Accel (a_x, a_y)
 [Memory] -> Global Coordinates [x, y, theta]
==================================================
                        |
                        v
  +--------------------------------------------+
  | Step 5.1: Motor Armature Current Update    |
  | di/dt = (V - R*i - Kb*omega) / Lm          |
  | i_{k+1} = i_k + dt * di/dt                 |
  | T_m = Kt * i_{k+1}                         |
  +--------------------------------------------+
                        |
                        v
  +--------------------------------------------+
  | Step 5.2: Dynamic Normal Force Distribution|
  | (x Right, y Forward Frame)                 |
  | Static load + Pitch (ay) + Roll (ax) + w_F |
  +--------------------------------------------+
                        |
                        v
  +--------------------------------------------+
  | Step 5.3: LuGre Bristle Friction Update    |
  | v_s = H*v_local - r*omega                  |
  | Exponential Discrete Update z_{k+1}        |
  | F_floor = - (mu_s*Fn*z + sigma_1*delta*z_dot) |
  +--------------------------------------------+
                        |
                        v
  +--------------------------------------------+
  | Step 5.4 & 5.5: Rigid Chassis Kinetics     |
  | a_local = M_inv * H^T * F_floor            |
  | v_local_{k+1} = v_local_k + dt * a_local   |
  | pos_global_{k+1} = pos_global + dt*R*v_local|
  +--------------------------------------------+
                        |
                        v
  +--------------------------------------------+
  | Step 5.6: Wheel Speed Integration          |
  | d_omega/dt = (T_m - T_roll - r*F_floor)/Jw |
  | omega_{k+1} = omega_k + dt * d_omega/dt    |
  +--------------------------------------------+
                        |
                        v
==================================================
        ADVANCE STATE TO STEP K+1
==================================================
```
</details>


#### `OmniRobotSimulationExplicit` Class Architecture


*   **`OmniRobotStateExplicit`**: Data container that preserves and moves integrated memory states (armature currents, wheel speeds, LuGre bristle deflections, local/global velocities, and position coordinates) from step $k$ to $k+1$.
*   **`__init__`**: Sets hardware constants, calculates lever arms ($x_f, x_b, y_f, y_b$), builds the $4 \times 3$ kinematic transformation matrix $\mathbf{H}$, and builds the inverse rigid-body mass matrix $\mathbf{M}^{-1}$.
*   **`_build_kinematics_matrix`**: Constructs the $4 \times 3$ matrix projecting 3-DOF body velocity into individual wheel rolling velocities.
*   **`compute_electrical_subsystem`**: Integrates motor currents $\mathbf{I}_{k+1}$ via Forward Euler and calculates gross motor torques $\mathbf{T}_{m,k}$.
*   **`compute_dynamic_normal_forces`**: Evaluates vertical normal loads $\mathbf{F}_{n,k}$ with pitch/roll load shifts based on past accelerations ($a_{x,\text{past}}, a_{y,\text{past}}$) and vibration noise.
*   **`compute_contact_friction_interface`**: Evaluates LuGre bristle deflection states $\mathbf{z}_{k+1}$ using an **unconditionally stable exponential discrete update**, computing continuous traction forces and smoothed rolling drag.
*   **`advance_chassis_kinetics`**: Integrates Newton-Euler rigid chassis motion and maps body velocities into global coordinates.
*   **`step`**: Executes the master pipeline loop for one discrete time step $\Delta T$.

In [3]:
import numpy as np

class OmniRobotStateExplicit:
    """
    Data container to log and preserve the electromechanical system states 
    at any discrete time step k.
    """
    def __init__(self):
        # Memory Integration States (Differential Cores)
        self.I = np.zeros(4)            # Armature currents [i1, i2, i3, i4] (A)
        self.omega = np.zeros(4)        # Wheel angular velocities [w1, w2, w3, w4] (rad/s)
        self.z = np.zeros(4)            # LuGre normalized bristle deflections [z1, z2, z3, z4]
        self.v_local = np.zeros(3)      # Local chassis velocities [vx (Right), vy (Forward), omega] (m/s, rad/s)
        self.a_local_past = np.zeros(3) # Historical chassis accelerations from step k-1 (m/s^2, rad/s^2)
        self.v_global = np.zeros(3)     # Global velocities [vx_g, vy_g, omega_g] (m/s, rad/s)
        self.pos_global = np.zeros(3)   # Global coordinates [x, y, theta] (m, rad)
        self.V_actual = np.zeros(4)     # Clipped applied terminal voltages [V1, V2, V3, V4] (V)
        
        # Tracked Logical Status Vector 
        self.S = np.zeros(4, dtype=bool) # Slip threshold indicators (False=Below slip band, True=Exceeds slip band)


class OmniRobotSimulationExplicit:
    """
    Physics engine plant managing an asymmetric 4-Omni wheel robot model
    with dynamic weight transfer and LuGre bristle floor friction boundaries.
    """
    def __init__(self, params: dict, random_seed: int = 42):
        """
        Initializes hardware constants, geometric parameters, and noise variances.
        """
        self.p = params
        self.dt = params['dt']
        
        # Pre-compute fixed kinematic tracking matrix (H)
        self.H = self._build_kinematics_matrix()

        # Dedicated random number generator state
        self.rng = np.random.default_rng(seed=random_seed)

        # --- BUILD THE MASS / INERTIA MATRIX (M) ---
        self.M = np.array([
            [self.p['Mr'], 0.0,           0.0],          # Mass acting on X acceleration (Right)
            [0.0,          self.p['Mr'],  0.0],          # Mass acting on Y acceleration (Forward)
            [0.0,          0.0,           self.p['Jzz']] # Rotational inertia acting on Alpha
        ])
        
        self.M_inv = np.linalg.inv(self.M)
        
        # Structural wheelbase lengths for weight distribution balances (+x Right, +y Forward)
        self.y_f = self.p['Rb'] * np.sin(self.p['alpha_f'])
        self.y_b = self.p['Rb'] * np.sin(self.p['alpha_b'])
        self.x_f = self.p['Rb'] * np.cos(self.p['alpha_f'])
        self.x_b = self.p['Rb'] * np.cos(self.p['alpha_b'])
        self.L = self.y_f + self.y_b
        self.sum_X2 = (self.x_f**2) + (self.x_b**2)
        self.m_rim = self.p['Jw'] / (self.p['r']**2)

    def _build_kinematics_matrix(self) -> np.ndarray:
        """Constructs the exact asymmetric velocity projection transformation matrix H."""
        af = self.p['alpha_f']
        ab = self.p['alpha_b']
        Rb = self.p['Rb']
        return np.array([
            [-np.sin(af),  np.cos(af), Rb],  # Wheel 1: Front Right
            [-np.sin(af), -np.cos(af), Rb],  # Wheel 2: Front Left
            [ np.sin(ab), -np.cos(ab), Rb],  # Wheel 3: Rear Left
            [ np.sin(ab),  np.cos(ab), Rb]   # Wheel 4: Rear Right
        ])

    def compute_electrical_subsystem(self, state: OmniRobotStateExplicit, V_input: np.ndarray) -> np.ndarray:
        """
        Calculates Step 5.1: Enforces Voltage Hardpoint Saturation Bounds (V_max) 
        and Over-Current Protection Hardpoint Bounds (I_max). Advances motor armature 
        currents using Forward Euler discrete integration and maps them to gross 
        electromagnetic torques.
        """
        Lm = self.p['Lm']
        R = self.p['R']
        Kb = self.p['Kb']
        Kt = self.p['Kt']
        V_max = self.p.get('V_max', 12.0)
        I_max = self.p.get('I_max', 20.0)
        
        # 1. Voltage Hardpoint Clipping (Power Supply / H-Bridge Saturation Limit)
        V_clipped = np.clip(V_input, -V_max, V_max)
        state.V_actual = V_clipped
        
        # 2. Back-EMF & Current Derivative Calculation
        v_b = Kb * state.omega
        di_dt = (V_clipped - R * state.I - v_b) / Lm
        
        # 3. Discrete Current Integration with Over-Current Hardpoint Clipping
        I_unclipped = state.I + self.dt * di_dt
        state.I_next = np.clip(I_unclipped, -I_max, I_max)
        
        # 4. Gross Electromagnetic Motor Torque Output
        T_m = Kt * state.I_next
        return T_m

    def compute_dynamic_normal_forces(self, state: OmniRobotStateExplicit) -> np.ndarray:
        """
        Calculates Step 5.2: Determines the vertical load profile for each wheel 
        by balancing static weight distributions with dynamic multi-axis pitch/roll 
        inertial moments (+x Right, +y Forward), plus random structural vibration noise.
        """
        #TODO: ADD diffrent friction per each wheel
        Mr = self.p['Mr']
        g = 9.81
        h = self.p['h']
        L = self.L
        x_f = self.x_f
        x_b = self.x_b
        y_f = self.y_f
        y_b = self.y_b
        sum_X2 = self.sum_X2
        sigma_V = self.p['sigma_V']
        
        ax_past = state.a_local_past[0]  # Right
        ay_past = state.a_local_past[1]  # Forward
        
        # Base Static Distributions
        F_n_front_static = (Mr * g * y_b) / (2.0 * L)
        F_n_rear_static  = (Mr * g * y_f) / (2.0 * L)
        
        # Dynamic Weight Shifts
        delta_Fn_y = (Mr * ay_past * h) / (2.0 * L)  # Pitch due to forward accel
        lateral_moment_scale = (Mr * ax_past * h) / L
        delta_Fn_x_front = (x_f / sum_X2) * lateral_moment_scale * y_b  # Roll
        delta_Fn_x_rear  = (x_b / sum_X2) * lateral_moment_scale * y_f
        
        w_F = self.rng.normal(0.0, sigma_V, 4) if sigma_V > 0 else np.zeros(4)
        
        F_n = np.zeros(4)
        F_n[0] = F_n_front_static - delta_Fn_y - delta_Fn_x_front + w_F[0] # Wheel 1: Front Right
        F_n[1] = F_n_front_static - delta_Fn_y + delta_Fn_x_front + w_F[1] # Wheel 2: Front Left
        F_n[2] = F_n_rear_static  + delta_Fn_y + delta_Fn_x_rear  + w_F[2] # Wheel 3: Rear Left
        F_n[3] = F_n_rear_static  + delta_Fn_y - delta_Fn_x_rear  + w_F[3] # Wheel 4: Rear Right
        
        return np.maximum(0.0, F_n)

    def compute_contact_friction_interface(self, T_m: np.ndarray, F_n: np.ndarray, state: OmniRobotStateExplicit) -> tuple:
        """
        Calculates Step 5.3: Evaluates LuGre bristle floor patch friction mechanics.
        Uses exact exponential discrete steps for stiff bristle deflection state.
        """
        r = self.p['r']
        Crr = self.p['Crr']
        mu_s_nom = self.p['mu_s']
        mu_k_nom = self.p['mu_k']
        sigma_mu = self.p['sigma_mu']
        delta_l = self.p['lugre_deflection']
        zeta = self.p['lugre_damping_ratio']
        v_sigma = self.p['lugre_damping_slip']
        v_c = self.p['lugre_creep']
        alpha_d = self.p['alpha_d']
        roll_smoothing = self.p['roll_smoothing']
        viscous = self.p['lugre_viscous']
        
        dt = self.dt
        
        w_mu = self.rng.normal(0.0, sigma_mu, 4) if sigma_mu > 0 else np.zeros(4)
        mu_s = np.maximum(0.0, mu_s_nom + w_mu)
        mu_k = np.maximum(0.0, mu_k_nom + w_mu)
        
        v_contact = self.H @ state.v_local
        v_s = v_contact - r * state.omega
        
        T_roll = Crr * F_n * r * np.tanh(state.omega / roll_smoothing)
        
        mu_d = mu_k + (mu_s - mu_k) * np.exp(-alpha_d * np.abs(v_s))
        denom = np.sqrt(v_s**2 + v_c**2)
        z_ss = (v_s * mu_d) / (mu_s * denom)
        tau = (delta_l * mu_d) / (mu_s * denom)
        
        exp_factor = np.exp(-dt / tau)
        z_next = state.z * exp_factor + z_ss * (1.0 - exp_factor)
        z_dot = (z_next - state.z) / dt
        
        sigma_1 = 2.0 * zeta * np.sqrt((mu_s * F_n / delta_l) * self.m_rim) * np.exp(-(v_s / v_sigma)**2)
        F_floor = -(mu_s * F_n * z_next + sigma_1 * delta_l * z_dot + viscous * v_s)
        
        T_load = -F_floor * r + T_roll
        
        return F_floor, T_load, z_next, T_roll

    def advance_chassis_kinetics(self, F_floor: np.ndarray, state: OmniRobotStateExplicit) -> OmniRobotStateExplicit:
        """
        Calculates Step 5.4 & 5.5: Advances rigid chassis velocities and global 
        position using Newton-Euler equations.
        """
        dt = self.dt
        theta_k = state.pos_global[2]
        
        a_local = self.M_inv @ (self.H.T @ F_floor)
        v_local_next = state.v_local + dt * a_local
        
        cos_t = np.cos(theta_k)
        sin_t = np.sin(theta_k)
        R_theta = np.array([
            [cos_t, -sin_t, 0.0],
            [sin_t,  cos_t, 0.0],
            [0.0,    0.0,   1.0]
        ])
        
        v_global_next = R_theta @ v_local_next
        pos_global_next = state.pos_global + dt * v_global_next
        
        next_state = OmniRobotStateExplicit()
        next_state.v_local = v_local_next
        next_state.a_local_past = a_local
        next_state.v_global = v_global_next
        next_state.pos_global = pos_global_next
        
        return next_state

    def step(self, current_state: OmniRobotStateExplicit, V_input: np.ndarray) -> tuple:
        """
        Executes the entire sequential discrete physics pipeline for a single time step dt.
        """
        r = self.p['r']
        Jw = self.p['Jw']
        dt = self.dt
        
        T_m = self.compute_electrical_subsystem(current_state, V_input)
        F_n = self.compute_dynamic_normal_forces(current_state)
        F_floor, T_load, z_next, T_roll = self.compute_contact_friction_interface(T_m, F_n, current_state)
        
        next_state = self.advance_chassis_kinetics(F_floor, current_state)
        next_state.I = current_state.I_next
        next_state.V_actual = current_state.V_actual
        next_state.z = z_next
        
        d_omega_dt = (T_m - T_roll - r * F_floor) / Jw
        next_state.omega = current_state.omega + dt * d_omega_dt
        
        return next_state
    
    def get_observable_state( self, full_state: OmniRobotStateExplicit ) -> OmniRobotObservableState: 
        obs = OmniRobotObservableState() 
        obs.pos_global = full_state.pos_global.copy() 
        obs.omega = full_state.omega.copy() 
        obs.I = full_state.I.copy() 
        obs.V_actual = full_state.V_actual.copy()
        return obs


## `OmniRobotSimulationODESolve` Plant Simulation Specification: Discrete-Time Open-Loop Asymmetric 4-Omni Robot Model

This cell specifies the comprehensive mathematical framework for the 4-wheel omni-directional mobile robot simulation. The architecture accounts for **asymmetric wheel layouts**, **dynamic multi-axis weight transfer (pitch and roll)**, **non-calibrated hardware parameters**, and **LuGre bristle floor friction boundaries** evaluated via **implicit (backward Euler) discrete-time updates** at a fixed sampling interval $\Delta T = 1\text{ ms}$ ($1\text{ kHz}$ loop rate).

---
<details>
<summary><b> Open this for the full specification of the simulation equations</b></summary>


### 1. Parameters & Variable Definitions

#### 🛠️ Hardware & Structural Parameters (Henry Calibrated Constants)
* $M_r$ : Total mass of the robot
* $J_{zz}$: Rotational mass moment of inertia around vertical Z-axis
* $J_w $: Rotational moment of inertia of a wheel/rotor assembly
* $R_b$ : Geometric base radius from the base centre to the wheel contact points
* $r $: Operating radius of omni-wheel tires
* $\alpha_f$ = : Front wheel mounting angle relative to local $+x_r$ axis
* $\alpha_b $= : Rear wheel mounting angle relative to local $+x_r$ axis
* $h$: Height of Center of Mass (CoM) above ground plane
* $y_{cg}$ : Center of Mass position forward of the base centre (0 for now: centred)
* $V_{max}$ : Supply voltage saturation limit
* $R$ , $L_m$ : DC motor armature winding resistance and inductance
* $K_t $= $K_b$ : Electromechanical torque and back-EMF constants
* $C_{rr}$ : Floor-tire rolling resistance coefficient
* $\mu_s$ , $\mu_k$ : Static and kinetic floor-tire friction coefficients
* $\delta$ : Presliding bristle deflection limit ($2\,\mu\text{m}$)
* $v_c $: Creep regularization velocity preventing division by zero at rest
* $\alpha_d $: Stribeck exponential decay rate from static to kinetic friction
* $\varepsilon $: Hyperbolic tangent smoothing speed for rolling resistance drag
* $\mu_{a,s}$, $\mu_{a,k}$, $\alpha_a$, $k_a$ : Roller (a-axis) friction coefficients at zero and at high axial slip, their decay rate, and the slope of the smoothed sign (change 9)
* $\epsilon_{tol}$, $n_{max}$ : Tolerance on the slip change, and the most solves per step, for the wheel solve (`implicit_tol`, `implicit_max_iter`)
* $\sigma_V$, $\sigma_\mu$ : Standard deviations of the wheel load noise $w_F$ and the floor friction noise $w_\mu$ (`sigma_V`, `sigma_mu`)

---

### 2. Signal Tracker Definitions

At each sample time step $k$, the discrete simulation cycle processes the following variables:

#### System Inputs
* $\mathbf{V}_k = [V_{1,k}, V_{2,k}, V_{3,k}, V_{4,k}]^\top$: Armature terminal voltage commands (Volts), clipped to $\pm V_{max}$.

#### Tracked Memory Integration States
* $\mathbf{I}_k = [i_{1,k}, i_{2,k}, i_{3,k}, i_{4,k}]^\top$: Motor armature currents (Amperes).
* $\mathbf{\Omega}_k = [\omega_{1,k}, \omega_{2,k}, \omega_{3,k}, \omega_{4,k}]^\top$: Wheel angular velocities (rad/s).
* $\mathbf{z}_k = [z_{1,k}, z_{2,k}, z_{3,k}, z_{4,k}]^\top$: Normalized bristle deflections (meters / normalized).
* $\mathbf{\dot{X}}_{r,k} = [\dot{x}_{r,k}, \dot{y}_{r,k}, \dot{\theta}_{k}]^\top$: Local chassis velocities (m/s Right, m/s Forward, rad/s Yaw), derived from the global velocities.
* $\mathbf{\ddot{X}}_{r,k-1} = [a_{x,r,k-1}, a_{y,r,k-1}, \alpha_{r,k-1}]^\top$: Historical chassis accelerations from step $k-1$ (m/s², m/s², rad/s²).
* $\mathbf{\dot{X}}_{g,k} = [\dot{x}_{g,k}, \dot{y}_{g,k}, \dot{\theta}_{g,k}]^\top$: Global room frame velocities (m/s, m/s, rad/s): the advanced velocity state, since Newton's law holds in this fixed frame.
* $\mathbf{X}_{g,k} = [x_k, y_k, \theta_k]^\top$: Global room frame navigation coordinates (meters, meters, radians).

---

### 3. Kinematic Mapping Matrices $\mathbf{H}_{drive}$, $\mathbf{H}_{axial}$ (Body Frame: $+x_r$ Right, $+y_r$ Forward)

The chassis motion is measured about the Center of Mass, $y_{cg}$ forward of the base centre; with $\mathbf{p}_i$ each wheel's contact point from it. Each row of a matrix is a wheel's direction followed by its yaw lever: $\mathbf{H}_{drive}$ has rows $[\mathbf{d}_i^\top,\ (\mathbf{p}_i \times \mathbf{d}_i)_z]$ and $\mathbf{H}_{axial}$ has rows $[\mathbf{a}_i^\top,\ (\mathbf{p}_i \times \mathbf{a}_i)_z]$:

The drive direction is the axial direction turned $90^\circ$:

$$\delta_f = \alpha_f + 90^\circ, \qquad \delta_b = \alpha_b + 90^\circ$$

$$\mathbf{v}_{w,k} = \mathbf{H}_{drive} \mathbf{\dot{X}}_{r,k}, \qquad \mathbf{H}_{drive} = \begin{bmatrix} \cos\delta_f & \sin\delta_f & R_b + y_{cg}\cos\delta_f \\ \cos\delta_f & -\sin\delta_f & R_b + y_{cg}\cos\delta_f \\ -\cos\delta_b & -\sin\delta_b & R_b - y_{cg}\cos\delta_b \\ -\cos\delta_b & \sin\delta_b & R_b - y_{cg}\cos\delta_b \end{bmatrix}$$

$$\mathbf{v}_{a,k} = \mathbf{H}_{axial} \mathbf{\dot{X}}_{r,k}, \qquad \mathbf{H}_{axial} = \begin{bmatrix} \cos\alpha_f & \sin\alpha_f & y_{cg}\cos\alpha_f \\ -\cos\alpha_f & \sin\alpha_f & -y_{cg}\cos\alpha_f \\ -\cos\alpha_b & -\sin\alpha_b & -y_{cg}\cos\alpha_b \\ \cos\alpha_b & -\sin\alpha_b & y_{cg}\cos\alpha_b \end{bmatrix}$$

$J_{zz}$ is the yaw inertia about the Center of Mass, so the mass matrix is $\mathbf{M} = \text{diag}(M_r, M_r, J_{zz})$.

---

### 4. Dynamic Normal Load Redistribution

With $+x_r$ Right (strafing) and $+y_r$ Forward (longitudinal):
$$y_f = R_b \sin\alpha_f = 0.03975\text{ m}, \qquad y_b = R_b \sin\alpha_b = 0.05622\text{ m}, \qquad L = y_f + y_b = 0.09597\text{ m}$$
$$x_f = R_b \cos\alpha_f = 0.06885\text{ m}, \qquad x_b = R_b \cos\alpha_b = 0.05622\text{ m}, \qquad \Sigma X^2 = x_f^2 + x_b^2 = 0.007901\text{ m}^2$$
The front and rear wheel distances from the Center of Mass (equal to $y_f$, $y_b$ while it is centred):
$$y_{f,cg} = y_f - y_{cg}, \qquad y_{b,cg} = y_b + y_{cg}$$

#### Static Normal Force Distribution at Rest (values with the CoG centred)
$$F_{n,\text{front, static}} = \frac{M_r g y_{b,cg}}{2 L} = 7.752\text{ N}, \qquad F_{n,\text{rear, static}} = \frac{M_r g y_{f,cg}}{2 L} = 5.482\text{ N}$$

#### Dynamic Weight Shifts (Pitch due to $a_{y,\text{past}}$, Roll due to $a_{x,\text{past}}$)
$$\Delta F_{n,y} = \frac{M_r a_{y,\text{past}} h}{2 L}, \qquad \Delta F_{n,x,\text{front}} = \frac{x_f}{\Sigma X^2} \frac{M_r a_{x,\text{past}} h y_{b,cg}}{L}, \qquad \Delta F_{n,x,\text{rear}} = \frac{x_b}{\Sigma X^2} \frac{M_r a_{x,\text{past}} h y_{f,cg}}{L}$$

$$\mathbf{F}_{n,k} = \begin{bmatrix} F_{n,\text{front, static}} - \Delta F_{n,y} - \Delta F_{n,x,\text{front}} \\ F_{n,\text{front, static}} - \Delta F_{n,y} + \Delta F_{n,x,\text{front}} \\ F_{n,\text{rear, static}} + \Delta F_{n,y} + \Delta F_{n,x,\text{rear}} \\ F_{n,\text{rear, static}} + \Delta F_{n,y} - \Delta F_{n,x,\text{rear}} \end{bmatrix}$$

The robot adds the load disturbance $w_{F,i,k} \sim \mathcal{N}(0, \sigma_V^2)$, drawn each step inside `compute_dynamic_normal_forces`, and a wheel cannot pull on the floor:
$$F_{n,i,k} \leftarrow \max\left(0,\ F_{n,i,k} + w_{F,i,k}\right)$$

---

### 5. Motor, Bristle Friction (Modified Dahl) & Discrete Integration

The motor current, wheel speed and bristle deflection of each wheel change faster than $\Delta T$ (the bristles' presliding and the motor's $L_m/R$), so they are discretised with **backward (implicit) Euler**, $\frac{dx}{dt} \approx \frac{x_{k+1} - x_k}{\Delta T}$ with the right-hand side at step $k+1$, which stays stable at any $\Delta T$. The three are solved together for each wheel (item 7). The chassis then follows with Forward Euler.

**Step 5.1: Supply Limit and Motor Armature Current:**
   $$V_{i,k} = \text{clip}\left(V_{cmd,i,k}, -V_{max}, V_{max}\right), \qquad L_m \frac{i_{i,k+1} - i_{i,k}}{\Delta T} = V_{i,k} - R\, i_{i,k+1} - K_b\, \omega_{i,k+1}, \qquad T_{m,i,k+1} = K_t\, i_{i,k+1}$$

**Step 5.2: Contact Slip Velocity along Rolling Axis** (chassis contact speed of step $k$, new wheel speed):
   $$v_{s,i,k+1} = (\mathbf{H}_{drive} \mathbf{\dot{X}}_{r,k})_i - r \omega_{i,k+1}$$

**Step 5.3: Smoothed Rolling Resistance Drag:**
   $$T_{\text{roll},i,k} = C_{rr} F_{n,i,k} r \tanh\left( \frac{\omega_{i,k}}{\varepsilon} \right)$$

**Step 5.4: Stribeck Friction Curve:**
   The friction coefficients include the floor friction noise $w_{\mu,i,k} \sim \mathcal{N}(0, \sigma_\mu^2)$, drawn once per step and held for every pass of the wheel solve (Step 5.7):
   $$\mu_{s,i} = \max\left(0,\ \mu_s + w_{\mu,i,k}\right), \qquad \mu_{k,i} = \max\left(0,\ \mu_k + w_{\mu,i,k}\right)$$
   $$\mu_{d,i} = \mu_{k,i} + (\mu_{s,i} - \mu_{k,i}) e^{-\alpha_d |v_{s,i,k+1}|}$$
   In Steps 5.5 to 5.7, $\mu_s$ is each wheel's $\mu_{s,i}$.

**Step 5.5: Backward Euler Bristle Update ($\Delta T = 1\text{ ms}$):**
   The LuGre bristle equation:
   $$\delta \, \dot{z}_i = v_{s,i} - \frac{\mu_s}{\mu_{d,i}} \sqrt{v_{s,i}^2 + v_c^2}\; z_i$$
   Over one step, with backward Euler:
   $$\delta \, \frac{z_{i,k+1} - z_{i,k}}{\Delta T} = v_{s,i,k+1} - \frac{\mu_s}{\mu_{d,i}} \sqrt{v_{s,i,k+1}^2 + v_c^2}\; z_{i,k+1}$$

**Step 5.6: Net Tractive Floor Force:**
   $$F_{i,k+1} = -\mu_s F_{n,i,k} z_{i,k+1}$$
   The full LuGre model has other terms (bristle damping and viscous friction). They are removed here: this is a simple model, and all it needs is the bristle deflection term, so that the force is not zero at zero slip. This makes it a modified Dahl model: LuGre with $\sigma_1 = \sigma_2 = 0$, as in Dahl, but keeping the Stribeck curve, where strict Dahl sets $\mu_s = \mu_d$ ([Altair: LuGre and Dahl friction](https://help.altair.com/hwdesktop/altair_help/topics/tools/bushing_model_friction_formulation_ms_r.htm)).
   In sliding the bristles settle within a step, $F_i = -\text{sign}(v_{s,i})\, \mu_{d,i} F_{n,i}$: the Coulomb–Stribeck law.

**Step 5.7: Unified Wheel Speed Update (solved with Steps 5.1, 5.5 and 5.6):**
   $$J_w \frac{\omega_{i,k+1} - \omega_{i,k}}{\Delta T} = T_{m,i,k+1} - T_{\text{roll},i,k} - r F_{i,k+1}$$
   With $v_{s,i,k+1} = v_{w,i,k} - r\,\omega_{i,k+1}$ (Step 5.2), $F_{i,k+1} = -\mu_s F_{n,i,k} z_{i,k+1}$ (Step 5.6) and $d_i = \frac{\mu_s}{\mu_{d,i}} \sqrt{v_{s,i}^2 + v_c^2}$ held fixed, the motor (Step 5.1), wheel (this step) and bristle (Step 5.5) equations are linear in the three new values. Moving those to the left gives one matrix equation per wheel, $\mathbf{A}_i \mathbf{x}_{i,k+1} = \mathbf{b}_i$, with one row per equation:
   $$\underbrace{\begin{bmatrix} L_m/\Delta T + R & K_b & 0 \\ -K_t & J_w/\Delta T & -r \mu_s F_{n,i,k} \\ 0 & r & \delta/\Delta T + d_i \end{bmatrix}}_{\mathbf{A}_i} \underbrace{\begin{bmatrix} i_{i,k+1} \\ \omega_{i,k+1} \\ z_{i,k+1} \end{bmatrix}}_{\mathbf{x}_{i,k+1}} = \underbrace{\begin{bmatrix} (L_m/\Delta T)\, i_{i,k} + V_{i,k} \\ (J_w/\Delta T)\, \omega_{i,k} - T_{\text{roll},i,k} \\ (\delta/\Delta T)\, z_{i,k} + v_{w,i,k} \end{bmatrix}}_{\mathbf{b}_i} \quad \begin{matrix} \text{motor} \\ \text{wheel} \\ \text{bristle} \end{matrix}$$
   The chassis is held during the solve, so the wheels do not affect each other. $d_i$ depends on the new slip through $\mu_{d,i}$, so the equation is solved again with $d_i$ updated at the new slip until the slip changes by less than $\epsilon_{tol}$ (at most $n_{max}$ solves). The load torque on each motor is $T_{load,i,k} = r F_{i,k+1} + T_{\text{roll},i,k}$.

**Step 5.8: Roller (a-axis) Friction — explicit, from the chassis velocity of step $k$ (change 9):** each wheel's rollers let it slide along its axial direction $\mathbf{a}_i$ (the drive direction $\mathbf{d}_i$ turned $-90^\circ$), and the roller bushings resist that sliding.:
   $$v_{a,i,k} = (\mathbf{H}_{axial} \mathbf{\dot{X}}_{r,k})_i, \qquad \mu_{a,i} = \mu_{a,k} + (\mu_{a,s} - \mu_{a,k})\, e^{-\alpha_a |v_{a,i,k}|}$$
   $$F_{a,i,k} = -F_{n,i,k}\, \mu_{a,i}\, \frac{2}{\pi} \arctan\left(k_a\, v_{a,i,k}\right), \qquad \mathbf{W}_{a,k} = \mathbf{H}_{axial}^\top \mathbf{F}_{a,k}$$
   The velocity of step $k$ is held over the step: a one-step lag, which is stable because this force is not stiff (it changes the chassis velocity at most about 160 /s, against $1/\Delta T$ = 1000 /s). The roller force lies along the wheel's axial, so it does not turn the wheel; it acts on the chassis only. **Off by default:** `roller_friction` in `simulation_parameters`.

**Step 5.9: Rigid Chassis Kinetics (global room frame):** the wheels are fixed to the chassis, so their forces act along body directions; Newton's law holds in the fixed room frame, so the global velocities are advanced there with the heading $\theta_k$:
   $$\mathbf{F}_{r,k} = \mathbf{H}_{drive}^\top \mathbf{F}_{k+1} + \mathbf{W}_{a,k}, \qquad \mathbf{F}_{g,k} = \mathbf{R}(\theta_k)\, \mathbf{F}_{r,k}, \qquad \mathbf{\ddot{X}}_{g,k} = \mathbf{M}^{-1} \mathbf{F}_{g,k}$$
   $$\mathbf{\dot{X}}_{g,k+1} = \mathbf{\dot{X}}_{g,k} + \Delta T\, \mathbf{\ddot{X}}_{g,k}, \qquad \mathbf{X}_{g,k+1} = \mathbf{X}_{g,k} + \Delta T\, \mathbf{\dot{X}}_{g,k+1}$$
   $$\mathbf{R}(\theta) = \begin{bmatrix} \cos\theta & -\sin\theta & 0 \\ \sin\theta & \cos\theta & 0 \\ 0 & 0 & 1 \end{bmatrix}, \qquad \mathbf{\dot{X}}_{r,k+1} = \mathbf{R}(\theta_{k+1})^\top \mathbf{\dot{X}}_{g,k+1}, \qquad \mathbf{\ddot{X}}_{r,k} = \mathbf{R}(\theta_k)^\top \mathbf{\ddot{X}}_{g,k}$$
   There is no separate rotation term: Newton's law is written in the fixed room frame, and the local velocity and acceleration are only the global ones rotated into the robot's axes. $\mathbf{\ddot{X}}_{r,k}$ is the global acceleration along the robot's axes, used by the load transfer of the next step; it is not the time derivative of $\mathbf{\dot{X}}_r$.

</details>

#### `OmniRobotSimulationODESolve` Class Architecture


*   **`OmniRobotStateODESolve`**: Data container that preserves and moves integrated memory states (armature currents, wheel speeds, LuGre bristle deflections, local/global velocities, and position coordinates) from step $k$ to $k+1$.
*   **`__init__`**: Sets hardware constants, calculates lever arms ($x_f, x_b, y_f, y_b$), builds the $4 \times 3$ kinematic transformation matrix $\mathbf{H}$, and builds the inverse rigid-body mass matrix $\mathbf{M}^{-1}$.
*   **`step`**: Executes the master pipeline loop for one discrete time step $\Delta T$.

In [4]:
"""
The robot
=========

OmniMotorState holds the full state of the robot at one step k. OmniRobotSimulation is the robot: it
is created once from simulation_parameters, works out everything fixed in __init__, and step advances
the full state by one dt. Several robots can run side by side, each with its own parameters and noise.

From outside, the robot is a black box:
    step(state, V_cmd)            advances the full state by one dt
    measure(state)                what the controller sees: [X, Y, theta, omega_1..4]

Inside, step calls:

    supply clip of the voltage commands
        ↓
    compute_dynamic_normal_forces        wheel loads, with the load noise
        ↓
    compute_roller_friction              roller friction, if switched on
        ↓
    compute_contact_friction_interface   friction noise, then motor, wheel and bristles solved together
        ↓
    advance_chassis_kinetics             chassis acceleration, velocity and pose (global frame)
"""

import numpy as np


class OmniRobotStateODESolve:
    """
    Data container that preserves the states of the robot at any discrete time step k.
    """
    def __init__(self):
        self.I = np.zeros(4)              # motor currents [A]
        self.omega = np.zeros(4)          # wheel speeds [rad/s]
        self.z = np.zeros(4)              # bristle deflections, as a fraction of lugre_deflection
        self.v_global = np.zeros(3)       # global velocity [x_dot, y_dot, theta_dot], the advanced state
        self.v_local = np.zeros(3)        # local velocity [x_dot, y_dot, theta_dot], derived from v_global
        self.pos_global = np.zeros(3)     # pose [X, Y, theta] [m, m, rad]
        self.a_local_past = np.zeros(3)   # chassis acceleration of the previous step, local frame [x, y, yaw]
        self.V_actual = np.zeros(4)       # motor voltages after the supply clip [V]

        # Slip flags: True where a wheel's slip is above slip_band (a report only)
        self.S = np.zeros(4, dtype=bool)


class OmniRobotSimulationODESolve:
    """
    The 4-omni-wheel robot: wheel layout, motors, floor friction and the rigid chassis.
    """

    # =======================================================================
    # Setup: everything fixed, worked out once
    # =======================================================================

    def __init__(self, params: dict, random_seed: int = 42):
        """
        params: simulation_parameters
        random_seed: seed of this robot's noise, so runs are repeatable
        """
        self.p = params.copy()
        self.dt = params['dt']

        # Dedicated random number generator for this robot's noise
        self.rng = np.random.default_rng(seed=random_seed)
        GRAVITY = 9.81  # m/s^2
        # -------------------------------------------------------------------
        # Wheel geometry (Section 3) [CHANGE 1, 6, 9]
        # -------------------------------------------------------------------
        alpha_f = params['alpha_f']
        alpha_b = params['alpha_b']
        Rb = params['Rb']
        cog_y = params['cog_y']

        # Wheel angles around the base circle, counter-clockwise from +x: 1 FR, 2 FL, 3 RL, 4 RR
        alpha = np.array([alpha_f, np.pi - alpha_f, np.pi + alpha_b, 2.0 * np.pi - alpha_b])

        # a_hat: the way the rollers slide (out from the centre). d_hat: the way the wheel pushes, 90 deg ahead
        self.a_hat = np.column_stack((np.cos(alpha), np.sin(alpha)))
        self.d_hat = np.column_stack((np.cos(alpha + np.pi / 2), np.sin(alpha + np.pi / 2)))

        # Each wheel's position measured from the CoG (r_i), the point the chassis equations are written about
        self.positions = Rb * self.a_hat - np.array([0.0, cog_y])

        # Yaw levers: z component of r_i x d_hat_i and r_i x a_hat_i, which is r_x d_y - r_y d_x
        r_x, r_y = self.positions[:, 0], self.positions[:, 1]
        drive_lever = r_x * self.d_hat[:, 1] - r_y * self.d_hat[:, 0]
        axial_lever = r_x * self.a_hat[:, 1] - r_y * self.a_hat[:, 0]

        # H_drive: rows are [drive x, drive y, yaw lever], so drive speeds = H_drive [x_dot, y_dot, theta_dot]
        # H_axial: rows are [axial x, axial y, yaw lever], so axial speeds = H_axial [x_dot, y_dot, theta_dot]
        self.H_drive = np.column_stack((self.d_hat, drive_lever))
        self.H_axial = np.column_stack((self.a_hat, axial_lever))
        self.H = self.H_drive  # the main kinematics matrix, used in the chassis equations
        # -------------------------------------------------------------------
        # Wheel loads: the fixed parts (Section 4)
        # -------------------------------------------------------------------
        Mr = params['Mr']
        h = params['h']

        x_front, y_front = self.positions[0]                            # wheel 1 FR, from the CoG
        x_rear, y_rear = self.positions[3][0], -self.positions[3][1]    # wheel 4 RR, distance behind the CoG

        wheelbase = y_front + y_rear
        sum_x2 = x_front**2 + x_rear**2

        # Static split: the front or rear wheels carry more when the CoG is closer to them
        self.F_n_front_static = (Mr * GRAVITY * y_rear) / (2.0 * wheelbase)
        self.F_n_rear_static = (Mr * GRAVITY * y_front) / (2.0 * wheelbase)

        # Pitch: load moved per unit of forward acceleration. One gain for all four wheels: the load
        # lost at the front equals the load gained at the rear, and each pair splits it equally
        # because the layout is assumed symmetric left to right.
        self.pitch_gain = (Mr * h) / (2.0 * wheelbase)

        # Roll: load moved per unit of sideways acceleration. Two gains: the front and rear wheels
        # are at different distances, so each pair carries a different share of the roll moment.
        roll_moment_gain = (Mr * h) / wheelbase
        self.roll_gain_front = (x_front / sum_x2) * roll_moment_gain * y_rear
        self.roll_gain_rear = (x_rear / sum_x2) * roll_moment_gain * y_front

        # -------------------------------------------------------------------
        # Mass matrix about the CoG: M = diag(Mr, Mr, Jzz)
        # -------------------------------------------------------------------
        self.M = np.diag([Mr, Mr, params['Jzz']])
        self.M_inv = np.linalg.inv(self.M)

        self.last_iterations = 0          # wheel solves the last step took (a report only)
        self.last_load_vibration = np.zeros(4)   # MERGED (Henry's IMU): load noise w_F of the last step [N] (read by the IMU)

    # =======================================================================
    # Frame rotation
    # =======================================================================

    @staticmethod
    def _rotation(theta: float) -> np.ndarray:
        """
        R(theta), which rotates [x, y, yaw] from the local frame into the global frame.
        Its transpose rotates from the global frame back into the local frame.
        """
        return np.array([[np.cos(theta), -np.sin(theta), 0.0],
                         [np.sin(theta),  np.cos(theta), 0.0],
                         [0.0,            0.0,           1.0]])

    # =======================================================================
    # Forces
    # =======================================================================

    def compute_dynamic_normal_forces(self, state: OmniRobotStateODESolve) -> np.ndarray:
        """
        Calculates Step 5.2: the vertical load on each wheel, from the static split, the pitch and roll
        load transfer of the previous step's acceleration, and the structural vibration noise.
        [CHANGE 1, 6]

        Using the previous step's acceleration avoids an algebraic loop: the loads set the forces, which
        set the acceleration.
        """
        sigma_V = self.p['sigma_V']

        ax_past = state.a_local_past[0]   # right
        ay_past = state.a_local_past[1]   # forward

        # Pitch: accelerating forward moves load from the front wheels to the rear
        pitch_shift = self.pitch_gain * ay_past

        # Roll: accelerating right moves load from the right wheels to the left
        roll_shift_front = self.roll_gain_front * ax_past
        roll_shift_rear = self.roll_gain_rear * ax_past

        # Structural vibration on the loads
        w_F = self.rng.normal(0.0, sigma_V, 4) if sigma_V > 0 else np.zeros(4)
        self.last_load_vibration = w_F   # MERGED (Henry's IMU): kept for the IMU (OmniRobotSimulationODESolveIMU)

        F_n = np.array([
            self.F_n_front_static - pitch_shift - roll_shift_front + w_F[0],   # 1 FR
            self.F_n_front_static - pitch_shift + roll_shift_front + w_F[1],   # 2 FL
            self.F_n_rear_static + pitch_shift + roll_shift_rear + w_F[2],     # 3 RL
            self.F_n_rear_static + pitch_shift - roll_shift_rear + w_F[3],     # 4 RR
        ])

        # A wheel cannot pull on the floor
        return np.maximum(0.0, F_n)

    def compute_roller_friction(self, state: OmniRobotStateODESolve, F_n: np.ndarray) -> np.ndarray:
        """
        Calculates Step 5.8: the roller (a-axis) friction wrench on the chassis, from the chassis
        velocity of this step's start. [CHANGE 9]

        It lies along each wheel's axial direction, so it cannot turn the wheel. Explicit, one step
        late: that is stable because this force is not stiff (at most about 160 /s against 1/dt = 1000 /s).

        Returns [force x, force y, moment] on the chassis about the CoG [N, N, N m]; zero when
        roller_friction is off.
        """
        if not self.p['roller_friction']:
            return np.zeros(3)

        mu_a_s = self.p['mu_a_s']
        mu_a_k = self.p['mu_a_k']
        alpha_a = self.p['alpha_a']
        k_a = self.p['roller_smoothing']

        # Axial slip of each wheel: v_slip_a = H_axial v_local (the wheel's spin adds nothing along its axle)
        v_slip_a = self.H_axial @ state.v_local

        # Roller friction coefficient: mu_a_s at zero axial slip, tending to mu_a_k
        mu_a = mu_a_k + (mu_a_s - mu_a_k) * np.exp(-alpha_a * np.abs(v_slip_a))

        # (2/pi) atan(k v) is a smoothed sign(v)
        F_a = -F_n * mu_a * (2.0 / np.pi) * np.arctan(k_a * v_slip_a)

        # The wheels' forces and their moments about the CoG
        return self.H_axial.T @ F_a

    # =======================================================================
    # Wheel solve: drive friction (modified Dahl), motor and wheel together
    # =======================================================================
    #
    # Each wheel has a bristle deflection z, as a fraction of lugre_deflection (delta). At zero slip the
    # bristles hold their load, so the force does not drop to zero. In steady sliding the force is the
    # Stribeck curve mu_d F_n. v_s is the drive slip and v_c the creep speed (lugre_creep).
    #
    #     mu_d = mu_k + (mu_s - mu_k) exp(-alpha_d |v_s|)
    #     delta dz/dt = v_s - (mu_s / mu_d) sqrt(v_s^2 + v_c^2) z
    #     F = -mu_s F_n z
    #
    # The full LuGre model has other terms (bristle damping and viscous friction). They are removed here:
    # all this model needs is the bristle deflection term, so that the force is not zero at zero slip.
    # This makes it a modified Dahl model: LuGre with sigma_1 = sigma_2 = 0, as in Dahl, but keeping the
    # Stribeck curve, where strict Dahl sets mu_s = mu_d. Reference:
    # https://help.altair.com/hwdesktop/altair_help/topics/tools/bushing_model_friction_formulation_ms_r.htm
    #
    # The bristles and the wheel form a stiff loop (w*dt ~ 3.8), so they are solved implicitly with
    # backward Euler. [CHANGE 5]

    def compute_contact_friction_interface(self, state: OmniRobotStateODESolve, V: np.ndarray, F_n: np.ndarray) -> tuple:
        """
        Calculates Steps 5.3 to 5.7: the floor friction noise, the rolling drag, then each wheel's motor
        current, wheel speed and bristle deflection solved together. [CHANGE 5, 7]

        The friction noise (Dana's floor friction variation) is drawn once per step and held for every
        pass of the solve. The Stribeck coefficient and the bristle relax rate depend on the new slip,
        so the solve is repeated with them updated until the slip settles.

        V: applied motor voltages [V]
        F_n: from compute_dynamic_normal_forces [N]

        Returns (I_next, omega_next, z_next, F_floor, T_roll, solves):
            F_floor: force of the floor on each wheel, along its drive direction [N]
            T_roll: rolling drag on each wheel [N m]
            solves: how many times the solve ran
        """
        dt = self.dt
        r = self.p['r']
        Lm = self.p['Lm']
        R = self.p['R']
        Kb = self.p['Kb']
        Kt = self.p['Kt']
        Jw = self.p['Jw']
        Crr = self.p['Crr']
        roll_smoothing = self.p['roll_smoothing']
        mu_s_nom = self.p['mu_s']
        mu_k_nom = self.p['mu_k']
        sigma_mu = self.p['sigma_mu']
        alpha_d = self.p['alpha_d']
        delta = self.p['lugre_deflection']
        v_c = self.p['lugre_creep']
        tol = self.p['implicit_tol']
        max_iter = self.p['implicit_max_iter']

        # Floor friction variation for this step
        w_mu = self.rng.normal(0.0, sigma_mu, 4) if sigma_mu > 0 else np.zeros(4)
        mu_s = np.maximum(0.0, mu_s_nom + w_mu)
        mu_k = np.maximum(0.0, mu_k_nom + w_mu)

        # Wheel centre velocity along each drive direction, held over the step
        v_center_d = self.H_drive @ state.v_local

        # Step 5.3: rolling drag, smoothed through zero speed
        T_roll = Crr * F_n * r * np.tanh(state.omega / roll_smoothing)

        I_next = np.zeros(4)
        omega_next = np.zeros(4)
        z_next = np.zeros(4)

        # Drive slip at the start of the step: v_s = H_drive v_local - r omega
        v_s = v_center_d - r * state.omega
        for solves in range(1, max_iter + 1):
            # Steps 5.4 to 5.6 at the latest slip: Stribeck curve, bristle limit and relax rate
            mu_d = mu_k + (mu_s - mu_k) * np.exp(-alpha_d * np.abs(v_s))
            F_static = mu_s * F_n
            bristle_decay = (mu_s / mu_d) * np.sqrt(v_s**2 + v_c**2)

            # Step 5.7: the motor, wheel and bristle equations of each wheel at the end of the step
            # (backward Euler), as the matrix equation A x = b with x = [I_next, omega_next, z_next]:
            #
            #     motor:    L (I_next - I) / dt = V - R I_next - Kb omega_next
            #     wheel:    J (omega_next - omega) / dt = Kt I_next - T_roll + r F_static z_next
            #     bristle:  delta (z_next - z) / dt = v_center_d - r omega_next - bristle_decay z_next
            #
            # Moving the unknowns to the left gives one row of the matrix per equation:
            #
            #     [ L/dt + R    Kb      0                        ] [ I_next     ]   [ L/dt I + V              ]
            #     [ -Kt         J/dt    -r F_static              ] [ omega_next ] = [ J/dt omega - T_roll     ]
            #     [ 0           r       delta/dt + bristle_decay ] [ z_next     ]   [ delta/dt z + v_center_d ]
            for i in range(4):
                A = np.array([
                    [Lm / dt + R, Kb, 0.0],                              # motor
                    [-Kt, Jw / dt, -r * F_static[i]],                    # wheel
                    [0.0, r, delta / dt + bristle_decay[i]],             # bristle
                ])
                b = np.array([
                    Lm / dt * state.I[i] + V[i],                         # motor
                    Jw / dt * state.omega[i] - T_roll[i],                # wheel
                    delta / dt * state.z[i] + v_center_d[i],             # bristle
                ])
                I_next[i], omega_next[i], z_next[i] = np.linalg.solve(A, b)

            # Stop once the new slip has stopped changing
            v_s_next = v_center_d - r * omega_next
            settled = np.max(np.abs(v_s_next - v_s)) < tol
            v_s = v_s_next
            if settled:
                break

        # Step 5.6: F = -mu_s F_n z
        F_floor = -F_static * z_next
        return I_next, omega_next, z_next, F_floor, T_roll, solves

    # =======================================================================
    # Chassis
    # =======================================================================

    def advance_chassis_kinetics(self, state: OmniRobotStateODESolve, F_floor: np.ndarray, wrench_roller: np.ndarray) -> OmniRobotStateODESolve:
        """
        Calculates Step 5.9: the chassis acceleration from Newton's law, then the global velocity and
        pose with forward Euler. [CHANGE 2]

        The wheels are fixed to the chassis, so their forces start in the local frame. They are rotated
        into the global frame, where Newton's law holds, and the global velocity is the advanced state.
        Advancing the local velocity instead leaves out the rotation of the local axes, and a robot that
        moves while it turns follows a wrong, curved path.

        Returns a new OmniMotorState with the chassis part filled in.
        """
        dt = self.dt
        theta = state.pos_global[2]
        R_theta = self._rotation(theta)

        # Sum of forces on the chassis, local frame, then rotated into the global frame
        F_local = self.H_drive.T @ F_floor + wrench_roller
        F_global = R_theta @ F_local

        # Newton's law about the CoG, global frame
        a_global = self.M_inv @ F_global

        # Forward Euler: velocity, then position
        v_global_next = state.v_global + dt * a_global
        pos_global_next = state.pos_global + dt * v_global_next

        next_state = OmniRobotStateODESolve()
        next_state.v_global = v_global_next
        next_state.pos_global = pos_global_next
        next_state.v_local = self._rotation(pos_global_next[2]).T @ v_global_next   # back into the local frame, new heading
        next_state.a_local_past = R_theta.T @ a_global                              # for the next step's load transfer
        return next_state

    # =======================================================================
    # One step, and what the controller sees
    # =======================================================================

    def step(self, state: OmniRobotStateODESolve, V_cmd: np.ndarray) -> tuple:
        """
        Executes the whole discrete pipeline for one time step dt, with the voltages and the noise held
        over the step.

        state: OmniMotorState at step k
        V_cmd: motor voltage commands [V]

        Returns (next_state, T_load, S_status):
            next_state: OmniMotorState at step k+1
            T_load: load torque on each motor [N m]
            S_status: wheels whose slip this step is above slip_band (a report only)
        """
        V_max = self.p['V_max']
        r = self.p['r']
        

        # Step 5.1: supply limit [CHANGE 8]
        V = np.clip(np.asarray(V_cmd, dtype=float), -V_max, V_max)

        F_n = self.compute_dynamic_normal_forces(state)
        wrench_roller = self.compute_roller_friction(state, F_n)
        I_next, omega_next, z_next, F_floor, T_roll, self.last_iterations = \
            self.compute_contact_friction_interface(state, V, F_n)

        next_state = self.advance_chassis_kinetics(state, F_floor, wrench_roller)
        next_state.I = I_next
        next_state.omega = omega_next
        next_state.z = z_next
        next_state.V_actual = V

        # Load torque on each motor
        T_load = F_floor * r + T_roll

        # Slip flags: drive slip v_s = H_drive v_local - r omega, with the new wheel speeds
        v_slip_d = self.H_drive @ state.v_local - r * omega_next
        S_status = np.abs(v_slip_d) > 0.05
        next_state.S = S_status

        return next_state

    def get_observable_state( self, full_state: OmniRobotStateODESolve) -> OmniRobotObservableState: 
        obs = OmniRobotObservableState() 
        obs.pos_global = full_state.pos_global.copy() 
        obs.omega = full_state.omega.copy() 
        obs.I = full_state.I.copy() 
        obs.V_actual = full_state.V_actual.copy()
        return obs

In [ ]:
"""
The robot with an IMU
=====================

OmniRobotSimulationODESolveIMU is the same robot (OmniRobotSimulationODESolve, unchanged) with an IMU at the CoG.
get_observable_state also returns what the IMU reads at the end of each step (call it once per step):
    accelerometer: the chassis acceleration of the step just taken, along the robot's axes [right, forward]
                   (the plant's a_local_past)
    gyro:          the yaw rate theta_dot

Accelerometer noise: the structural vibration that shakes the wheel loads (sigma_V) also shakes the IMU, so the two
rise and fall together. The step's own load noise w_F (one value per wheel, N) is turned into the acceleration that
would have shifted the loads that way, through the load transfer matrix LT (F_n = F_static + LT a):
    a_noise = imu_vibration_gain * pinv(LT) w_F
pinv(LT) keeps the part of w_F that looks like pitch and roll. The rest (all four wheels bouncing together, or a
twist) is vertical, and the IMU's two planar axes do not see it. imu_vibration_gain sets how strongly the IMU picks
the vibration up (1: as much as the loads imply).

Gyro noise: its own white noise, sigma_gyro, from its own random generator, so the robot's other noise (and so its
motion) is the same as OmniRobotSimulationODESolve's with the same seed.

Not modelled: accelerometer and gyro bias, and the IMU sitting away from the CoG (it would also read the turning
accelerations of its offset).

Switch: imu_on (simulation parameters, True by default). False: the robot has no IMU, and get_observable_state
returns the readings without it (OmniRobotSimulationODESolve's).
"""

import numpy as np


class OmniRobotObservableStateIMU(OmniRobotObservableState):
    """
    What the controller observes at every time step, plus the IMU.
    """
    def __init__(self):
        super().__init__()
        self.imu_accel = np.zeros(2)    # Accelerometer [right, forward] (m/s^2)
        self.imu_yaw_rate = 0.0         # Gyro (rad/s)


class OmniRobotSimulationODESolveIMU(OmniRobotSimulationODESolve):
    """
    The ODESolve robot with an IMU at the CoG. The physics is the parent's, unchanged.
    """
    def __init__(self, params: dict, random_seed: int = 42):
        super().__init__(params, random_seed)

        # The gyro's own random generator, so the parent's noise sequence is not changed
        self.imu_rng = np.random.default_rng(seed=random_seed + 1)

        # Load transfer matrix (plant Step 5.2): load moved per unit acceleration [right, forward].
        # Rows: wheels 1 FR, 2 FL, 3 RL, 4 RR
        LT = np.array([
            [-self.roll_gain_front, -self.pitch_gain],     # 1 FR
            [ self.roll_gain_front, -self.pitch_gain],     # 2 FL
            [ self.roll_gain_rear,   self.pitch_gain],     # 3 RL
            [-self.roll_gain_rear,   self.pitch_gain],     # 4 RR
        ])
        # (2, 4): the acceleration that best explains a set of load changes
        self.LT_pinv = np.linalg.pinv(LT)

    def get_observable_state(self, full_state: OmniRobotStateODESolve) -> OmniRobotObservableStateIMU:
        # The IMU switch: off, the readings without the IMU
        if not self.p['imu_on']:
            return super().get_observable_state(full_state)

        obs = OmniRobotObservableStateIMU()
        obs.pos_global = full_state.pos_global.copy()
        obs.omega = full_state.omega.copy()
        obs.I = full_state.I.copy()
        obs.V_actual = full_state.V_actual.copy()

        # Accelerometer: the last step's acceleration, plus the same step's load vibration seen as an acceleration
        accel_noise = self.p['imu_vibration_gain'] * self.LT_pinv @ self.last_load_vibration
        obs.imu_accel = full_state.a_local_past[0:2] + accel_noise

        # Gyro: the yaw rate, plus its own noise
        sigma_gyro = self.p['sigma_gyro']
        gyro_noise = self.imu_rng.normal(0.0, sigma_gyro) if sigma_gyro > 0 else 0.0
        obs.imu_yaw_rate = full_state.v_global[2] + gyro_noise
        return obs

## Simulation Paramaters 
Any model paramater must be added to the paramater dictionary since this one dictionary gets passed to all plant models

In [5]:
# 1. Define the simulation parameters dictionary (Henry calibrated constants)
simulation_parameters = {
    # Time Domain Constraints
    'dt': 0.001,               # Loop sampling interval (1 kHz loop rate)

    # System Power & Actuator Hardpoint Limits
    'V_max': 12.0,             # Terminal supply voltage saturation limit (Volts)
    'I_max': 20.0,             # Motor driver over-current protection limit (Amperes)

    # Robot Mechanical Properties
    'Mr': 2.698,               # Total mass of the robot chassis (kg)
    'Jzz': 0.007726,           # Chassis rotational inertia around Z-axis (kg*m^2)
    'Rb': 0.0795,              # Structural chassis base radius (meters)
    'h': 0.0611,               # Height of the Center of Mass above ground (meters)

    # Omni-Wheel Geometry
    'r': 0.0299,               # Operating radius of individual omni-wheels (meters)
    'Jw': 0.00018,          # Reflected rotor assembly inertia for Pololu 37D 30:1 (kg*m^2)           # Rotational inertia of a wheel+motor assembly (kg*m^2)
    'alpha_f': np.radians(30.0),# Front wheels mounting angle (converted to radians)
    'alpha_b': np.radians(45.0),# Rear wheels mounting angle (converted to radians)

    # DC Motor Electromechanical Properties
    'R': 2.18,                 # Armature winding resistance for Pololu 37D (Ohms)                 # Armature winding resistance (Ohms)
    'Lm': 0.0023,               # Armature winding inductance for Pololu 37D (Henries, 2.3 mH)            # Armature winding inductance (Henries)
    'Kt': 0.255,                # Torque constant for Pololu 37D 30:1 (Nm/A)              # Torque constant (Nm/A)
    'Kb': 0.255,                # Back-EMF constant for Pololu 37D 30:1 (V/(rad/s))              # Back-EMF constant (V/(rad/s))

    # Surface Contact and Friction Parameters
    'Crr': 0.025,              # Coefficient of rolling resistance
    'mu_s': 0.70,              # Nominal static coefficient of floor friction
    'mu_k': 0.30,             # Nominal kinetic coefficient of floor friction

    # LuGre Friction Model Parameters
    'lugre_deflection': 2e-6,  # Presliding bristle deflection limit (m)
    'lugre_damping_ratio': 1.0,# Damping ratio fraction
    'lugre_damping_slip': 0.005,# Damping fade-out slip speed (m/s)
    'lugre_creep': 0.0001,     # Regularization velocity (m/s)
    'alpha_d': 5.0,            # Stribeck decay scale (s/m)
    'roll_smoothing': 0.5,     # Rolling resistance tanh smoothing speed (rad/s)
    'lugre_viscous': 0.0,      # Floor viscous damping (N*s/m)
    

    # Environmental Disturbance Variations (Stochastic Noise)
    'sigma_V': 0.15,          # Structural vibration standard deviation (Newtons). Henry's 0.15 (Dana's 2.0: the IMU's accelerometer then reads ~1.3 m/s^2 of noise)
    'sigma_mu': 0.03,          # Floor patch friction variance standard deviation. Henry's 0.03 (Dana's 0.1)
    'imu_on': True,            # IMU (Henry's): on/off switch. False: the robot has no IMU
    'imu_vibration_gain': 1.0, # IMU: how strongly the accelerometer picks up the load vibration (1: as much as the loads imply)
    'sigma_gyro': 0.002,       # IMU: gyro noise standard deviation (rad/s); a placeholder, of the order of a MEMS gyro at 1 kHz

    # Roller (a-axis) Friction 
    'roller_friction': False,        # On/off switch. False = free rollers
    'mu_a_s': 0.2,                   # Roller friction coefficient at zero axle slip
    'mu_a_k': 0.322,                 # Roller friction coefficient at high axle slip
    'alpha_a': 1.0,                  # Decay from mu_a_s to mu_a_k with axle slip speed (s/m)
    'roller_smoothing': 200.0,       # Slope k_a of the smoothed sign, (2/pi) atan(k_a v_a) (s/m)

    # Wheel Solver, backward Euler 
    'implicit_tol': 1e-4,            # Repeat the wheel solve until the slip changes by less than this (m/s)
    'implicit_max_iter': 50,         # Most wheel solves per step

    #variables only used in solver plant
    'cog_y': 0.0,                    # Center of Mass forward of the base centre (m). Centred for now; CAD value 0.0102
    
}

SIMULATION_PARAMATERS=simulation_parameters.copy()

## Testing Model Initialization

In [6]:
# 2. Instantiate the physics engine plant
robot_plant = OmniRobotSimulationExplicit(simulation_parameters, random_seed=42)

# 3. Instantiate a fresh initial state container
initial_state = OmniRobotStateExplicit()

# 4. Generate the Comprehensive Plant Status Report
print("=======================================================================")
print("                OMNI-WHEEL PHYSICAL PLANT INITIALIZATION REPORT        ")
print("=======================================================================")
print(f" TIME INTERRUPT  | Sampling Rate: {1.0/robot_plant.dt:.0f} Hz | dt: {robot_plant.dt} s")
print("-----------------------------------------------------------------------")
print(" CHASSIS MECHANICAL PROPERTIES:")
print(f"  • Total Vehicle Mass (Mr):        {robot_plant.p['Mr']:.3f} kg")
print(f"  • Center of Mass Height (h):      {robot_plant.p['h']:.3f} m")
print(f"  • Rotational Inertia (Jzz):       {robot_plant.p['Jzz']:.5f} kg·m²")
print(f"  • Structural Base Radius (Rb):    {robot_plant.p['Rb']:.4f} m")
print(f"  • Front Mounting Angle (α_f):     {np.degrees(robot_plant.p['alpha_f']):.1f}° ")
print(f"  • Rear  Mounting Angle (α_b):     {np.degrees(robot_plant.p['alpha_b']):.1f}° ")
print("-----------------------------------------------------------------------")
print(" ELECTROMECHANICAL DRIVE & HARDPOINT BOUNDS:")
print(f"  • Supply Voltage Limit (V_max):   {robot_plant.p['V_max']:.1f} V      | Over-Current Limit (I_max): {robot_plant.p['I_max']:.1f} A")
print(f"  • Winding Resistance (R):         {robot_plant.p['R']:.3f} Ω      | Winding Inductance (Lm):    {robot_plant.p['Lm']:.6f} H")
print(f"  • Torque Scale Constant (Kt):     {robot_plant.p['Kt']:.4f} Nm/A  | Back-EMF Scale (Kb):        {robot_plant.p['Kb']:.4f} V/(rad/s)")
print(f"  • Assembly Rotor Inertia (Jw):    {robot_plant.p['Jw']:.7f} kg·m² | Operating Tire Radius (r):  {robot_plant.p['r']:.4f} m")
print("-----------------------------------------------------------------------")
print(" ENVIRONMENTAL BOUNDARY CONSTRAINTS & STOCHASTIC DISTURBANCES:")
print(f"  • Base Rolling Friction Drag (Crr):   {robot_plant.p['Crr']:.4f}  | Vibrational Force Noise (σ_V): {robot_plant.p['sigma_V']:.3f} N")
print(f"  • Nominal Sliding Friction (μ_k):     {robot_plant.p['mu_k']:.3f} | Nominal Static Friction (μ_s): {robot_plant.p['mu_s']:.3f}")
print("=======================================================================")


                OMNI-WHEEL PHYSICAL PLANT INITIALIZATION REPORT        
 TIME INTERRUPT  | Sampling Rate: 1000 Hz | dt: 0.001 s
-----------------------------------------------------------------------
 CHASSIS MECHANICAL PROPERTIES:
  • Total Vehicle Mass (Mr):        2.698 kg
  • Center of Mass Height (h):      0.061 m
  • Rotational Inertia (Jzz):       0.00773 kg·m²
  • Structural Base Radius (Rb):    0.0795 m
  • Front Mounting Angle (α_f):     30.0° 
  • Rear  Mounting Angle (α_b):     45.0° 
-----------------------------------------------------------------------
 ELECTROMECHANICAL DRIVE & HARDPOINT BOUNDS:
  • Supply Voltage Limit (V_max):   12.0 V      | Over-Current Limit (I_max): 20.0 A
  • Winding Resistance (R):         2.180 Ω      | Winding Inductance (Lm):    0.002300 H
  • Torque Scale Constant (Kt):     0.2550 Nm/A  | Back-EMF Scale (Kb):        0.2550 V/(rad/s)
  • Assembly Rotor Inertia (Jw):    0.0001800 kg·m² | Operating Tire Radius (r):  0.0299 m
-------------------

#  Kinematic & Dynamic Verification Testing

This section executes open-loop verification trials to validate the electromechanical motor re-coupling, asymmetric kinematics mapping ($\mathbf{H}$), and friction boundaries under hardware-verified sign directions.

###  Structural Testing Grid Architecture
The execution pipeline performs a **multi-block comparative analysis** across two distinct physical frame designs and environmental settings:
1. **Block 1: Symmetric Frame Verification:** Configures the plant layout to an identical alignment across all wheel lines ($\alpha_f = \alpha_b$).
2. **Block 2: Asymmetric Frame Verification:** Evaluates the custom asymmetric chassis layout ($\alpha_f \ne \alpha_b$).
3. **Block 3: High Voltage Verification:** Increased the voltage input to each wheel in the asymmetric verification to indure greater slipping.

###  Verification Test Input Sequences
To isolate each independent degree of freedom, the simulation applies directional step voltage polarity patterns across the four wheel motors:
*   **Test 1: Forward Motion (Direction: $[+, -, -, +]^\top$):** Applies positive polarity to the right-side motors ($1, 4$) and negative polarity to the left-side motors ($2, 3$). This drives all wheels to push the ground backward symmetrically, canceling out lateral side forces and accelerating the robot forward along its local $+y_r$ axis.
*   **Test 2: Lateral Side-Strafing (Direction: $[-, -, +, +]^\top$):** Applies negative polarity to the front axle motors ($1, 2$) and positive polarity to the rear axle motors ($3, 4$). This mirrors longitudinal forces across the front-to-back split line, canceling out forward/backward vectors and accelerating the chassis sideways to the right along its local $+x_r$ axis.
*   **Test 3: Pure Rotational Spinning (Direction: $[+, +, +, +]^\top$):** Applies uniform positive polarity to all four wheels simultaneously. Because every omni-wheel is mounted tangentially to the chassis base circle, spinning them all in the same rotational direction eliminates all linear translation vectors and creates a pure angular torque, spinning the robot counter-clockwise ($+\theta$) around its Center of Mass.

###  Dual-Pass State Conditions
To clearly isolate the degradation caused by environmental disturbances, each vehicle geometry executes a sequential dual-pass simulation:
*   **Ideal Pass (Reference Baseline):** Strips all noise configurations ($\sigma_V = 0$, $\sigma_\mu = 0$). The results render as soft, transparent curves representing perfect theoretical trajectories.
*   **Perturbed Pass (Real-World Overlay):** Activates a repeatable, seeded random number generator stream (`np.random.default_rng`) to overlay high-frequency structural chassis vibrations ($w_F$) and spatial friction coefficient patches ($w_\mu$).

###  Logged Telemetry Variables
Each configuration runs the three directional maneuvers back-to-back over a time window of duration $t_{\text{end}}$ and generates a standalone 3x2 grid capturing:
*   **Applied Motor Terminal Voltages ($V_{1\dots4}$):** Verifies step-input directional polarities.
*   **Wheel Angular Velocities ($\omega_{1\dots4}$):** Monitors electromechanical back-EMF variations and independent rotor spinout integrations ($J_w$).
*   **Chassis Local Frame Speeds ($\dot{x}_r, \dot{y}_r, \dot{\theta}$):** Evaluates translational and angular momentum balances inside the mass matrix universe ($\mathbf{M}^{-1}\mathbf{H}^\top\mathbf{F}_k$).
*   **Global Position Tracking ($X-Y$ Trajectory Plane):** Visualizes dead-reckoning coordinate transformation drift paths directly overlaying ideal performance vs. noisy tracking realities.
*   **Wheel Contact Slip Velocity ($v_{s,1\dots4}$):** Displays individual contact patch slip velocities alongside the parameterized threshold band $(\pm v_{s,\text{limit}})$.
*   **Wheel Active Sliding Status Indicator:** Evaluates binary traction states ($|v_s| > v_{s,\text{limit}}$) to pinpoint exact sliding intervals.


In [7]:
#### CHOOOSING THE MODEL TO RUN and RUN PARAMATERS
model_choice = 2 # 1 for explicit, 2 for Linear System of Equation solver
low_voltage = 1.5
t_end = 1.5  
bost_voltage = 6.0
slip_limit=0.1
run_OpenLoop_Simulation = False

In [8]:
import numpy as np
import matplotlib.pyplot as plt

if run_OpenLoop_Simulation:
    nominal_sigma_V = simulation_parameters['sigma_V']
    nominal_sigma_mu = simulation_parameters['sigma_mu']

    dt = simulation_parameters['dt']
    steps = int(t_end / dt)
    t_vec = np.linspace(0, t_end, steps)

    maneuvers = [
        {"name": "Test 1: Forward Motion", "V": np.array([low_voltage, -low_voltage, -low_voltage, low_voltage])},
        {"name": "Test 2: Lateral Side-Strafing", "V": np.array([-low_voltage, -low_voltage, low_voltage, low_voltage])},
        {"name": "Test 3: Pure Rotational Spinning", "V": np.array([low_voltage, low_voltage, low_voltage, low_voltage])}
    ]

    # style modifiers
    style_ideal = {'alpha': 0.4, 'lw': 2.5}  
    style_noisy = {'alpha': 1.0, 'lw': 1.2}  

    # CONFIGURATION CHANGE: FORCE SYMMETRIC 45-45 DESIGN
    simulation_parameters['alpha_f'] = np.radians(45.0)
    simulation_parameters['alpha_b'] = np.radians(45.0)

    print("Executing Block 1: Symmetric 45-45 Wheel Angle Configurations...")

    for m in maneuvers:
        runs_data = {}
        for condition in ["Ideal", "With Disturbances"]:
            if condition == "Ideal":
                simulation_parameters['sigma_V'] = 0.0
                simulation_parameters['sigma_mu'] = 0.0
            else:
                simulation_parameters['sigma_V'] = nominal_sigma_V
                simulation_parameters['sigma_mu'] = nominal_sigma_mu

            if model_choice == 1:
                robot_plant = OmniRobotSimulationExplicit(simulation_parameters, random_seed=42)
                state = OmniRobotStateExplicit()    
            elif model_choice == 2:   
                robot_plant = OmniRobotSimulationODESolve(simulation_parameters, random_seed=42)
                state = OmniRobotStateODESolve()
            
            v_in_hist, omega_hist, v_local_hist, pos_global_hist = [], [], [], []
            
            for k in range(steps):
                v_in_hist.append(m["V"].copy())
                omega_hist.append(state.omega.copy())
                v_local_hist.append(state.v_local.copy())
                pos_global_hist.append(state.pos_global.copy())
                state = robot_plant.step(state, m["V"])
                
            omega_arr = np.array(omega_hist)
            v_local_arr = np.array(v_local_hist)
            v_s_arr = (v_local_arr @ robot_plant.H.T) - robot_plant.p['r'] * omega_arr
            runs_data[condition] = {
                "V": np.array(v_in_hist), "omega": omega_arr,
                "v_local": v_local_arr, "pos_global": np.array(pos_global_hist),
                "v_s": v_s_arr
            }
            
        ideal = runs_data["Ideal"]
        noisy = runs_data["With Disturbances"]

        fig, axs = plt.subplots(3, 2, figsize=(15, 15))
        fig.suptitle(f"Symmetric Block (45-45 Alignment) | {m['name']}", fontsize=16, fontweight='bold')
        
        # Subplot 1: Voltage Inputs
        for w in range(4): 
            axs[0, 0].plot(t_vec, ideal["V"][:, w], label=f'V{w+1}')
        axs[0, 0].set_title("Applied Motor Terminal Voltages"); axs[0, 0].set_xlabel("Time (s)"); axs[0, 0].set_ylabel("Voltage (V)"); axs[0, 0].grid(True); axs[0, 0].legend(loc="upper right")
        
        # Subplot 2: Wheel Speeds (FIXED: Raw string notation used)
        for w in range(4):
            axs[0, 1].plot(t_vec, ideal["omega"][:, w], '-', color=f'C{w}', label=r'$\omega_{' + str(w+1) + r'}$ Ideal', **style_ideal)
            axs[0, 1].plot(t_vec, noisy["omega"][:, w], '-', color=f'C{w}', label=r'$\omega_{' + str(w+1) + r'}$ Noise', **style_noisy)
        axs[0, 1].set_title("Wheel Angular Velocities Comparison"); axs[0, 1].set_xlabel("Time (s)"); axs[0, 1].set_ylabel("Speed (rad/s)"); axs[0, 1].grid(True); axs[0, 1].legend(loc="upper right", ncol=2, fontsize='small')
        
        # Subplot 3: Local Frame Chassis Velocities (FIXED: Isolated style configuration)
        axs[1, 0].plot(t_vec, ideal["v_local"][:, 0], 'r-', label=r'$\dot{x}_r$ Ideal', **style_ideal)
        axs[1, 0].plot(t_vec, noisy["v_local"][:, 0], 'r-', label=r'$\dot{x}_r$ Noise', **style_noisy)
        axs[1, 0].plot(t_vec, ideal["v_local"][:, 1], 'b-', label=r'$\dot{y}_r$ Ideal', **style_ideal)
        axs[1, 0].plot(t_vec, noisy["v_local"][:, 1], 'b-', label=r'$\dot{y}_r$ Noise', **style_noisy)
        axs[1, 0].plot(t_vec, ideal["v_local"][:, 2], 'g-', label=r'$\dot{\theta}$ Ideal', **style_ideal)
        axs[1, 0].plot(t_vec, noisy["v_local"][:, 2], 'g-', label=r'$\dot{\theta}$ Noise', **style_noisy)
        axs[1, 0].set_title("Chassis Local Frame Speeds"); axs[1, 0].set_xlabel("Time (s)"); axs[1, 0].set_ylabel("Velocity (m/s or rad/s)"); axs[1, 0].grid(True); axs[1, 0].legend(loc="upper right", ncol=2, fontsize='small')
        
        # Subplot 4: Global Trajectory Plane
        axs[1, 1].plot(ideal["pos_global"][:, 0], ideal["pos_global"][:, 1], 'k--', lw=2.0, label='Ideal Base Path')
        axs[1, 1].plot(noisy["pos_global"][:, 0], noisy["pos_global"][:, 1], 'b-', lw=1.5, label='Perturbed Path')
        axs[1, 1].plot(noisy["pos_global"][0, 0], noisy["pos_global"][0, 1], 'go', markersize=8, label='Start')
        axs[1, 1].plot(noisy["pos_global"][-1, 0], noisy["pos_global"][-1, 1], 'rx', markersize=8, label='End')
        axs[1, 1].set_title("Global Spatial Position Tracking (X-Y)"); axs[1, 1].set_xlabel("Global X Position (m)"); axs[1, 1].set_ylabel("Global Y Position (m)"); axs[1, 1].set_aspect('equal', 'datalim'); axs[1, 1].grid(True); axs[1, 1].legend(loc="best")
        
        # Subplot 5: Wheel Contact Slip Velocity (Option 2)
        for w in range(4):
            axs[2, 0].plot(t_vec, ideal["v_s"][:, w], color=f'C{w}', label=f'$v_{{s,{w+1}}}$ Ideal', **style_ideal)
            axs[2, 0].plot(t_vec, noisy["v_s"][:, w], color=f'C{w}', label=f'$v_{{s,{w+1}}}$ Noise', **style_noisy)
        axs[2, 0].axhline(slip_limit, color='r', linestyle='--', alpha=0.7, label=f'Sliding Limit (±{slip_limit*1000:.1f} mm/s)')
        axs[2, 0].axhline(-slip_limit, color='r', linestyle='--', alpha=0.7)
        axs[2, 0].set_title("Wheel Contact Slip Velocity (v_s in m/s)")
        axs[2, 0].set_xlabel("Time (s)"); axs[2, 0].set_ylabel("Slip Velocity (m/s)")
        axs[2, 0].grid(True); axs[2, 0].legend(loc="upper right", ncol=2, fontsize='small')

        # Subplot 6: Wheel Active Sliding Binary Indicator (|v_s| > 5 mm/s)
        for w in range(4):
            axs[2, 1].plot(t_vec, (np.abs(ideal["v_s"][:, w]) > slip_limit).astype(float), color=f'C{w}', label=f'Wheel {w+1} Sliding')
        axs[2, 1].set_title(f"Wheel Active Sliding Status Indicator (|v_s| > {slip_limit*1000:.1f} mm/s)")
        axs[2, 1].set_xlabel("Time (s)"); axs[2, 1].set_ylabel("Sliding Flag (1=Sliding, 0=Rolling)")
        axs[2, 1].set_ylim(-0.1, 1.1)
        axs[2, 1].grid(True); axs[2, 1].legend(loc="upper right", ncol=2, fontsize='small')

        plt.tight_layout()
        plt.show()


In [9]:
if run_OpenLoop_Simulation:

    # CONFIGURATION CHANGE: REVERT TO ASYMMETRIC 30-45 DESIGN
    simulation_parameters['alpha_f'] = np.radians(30.0)
    simulation_parameters['alpha_b'] = np.radians(45.0)

    print("Executing Block 2: Asymmetric 30-45 Wheel Angle Configurations...")

    for m in maneuvers:
        runs_data = {}
        for condition in ["Ideal", "With Disturbances"]:
            if condition == "Ideal":
                simulation_parameters['sigma_V'] = 0.0
                simulation_parameters['sigma_mu'] = 0.0
            else:
                simulation_parameters['sigma_V'] = nominal_sigma_V
                simulation_parameters['sigma_mu'] = nominal_sigma_mu
                
            if model_choice == 1:
                robot_plant = OmniRobotSimulationExplicit(simulation_parameters, random_seed=42)
                state = OmniRobotStateExplicit()    
            elif model_choice == 2:   
                robot_plant = OmniRobotSimulationODESolve(simulation_parameters, random_seed=42)
                state = OmniRobotStateODESolve()
                    
            
            v_in_hist, omega_hist, v_local_hist, pos_global_hist = [], [], [], []
            
            for k in range(steps):
                v_in_hist.append(m["V"].copy())
                omega_hist.append(state.omega.copy())
                v_local_hist.append(state.v_local.copy())
                pos_global_hist.append(state.pos_global.copy())
                state = robot_plant.step(state, m["V"])
                
            omega_arr = np.array(omega_hist)
            v_local_arr = np.array(v_local_hist)
            v_s_arr = (v_local_arr @ robot_plant.H.T) - robot_plant.p['r'] * omega_arr
            runs_data[condition] = {
                "V": np.array(v_in_hist), "omega": omega_arr,
                "v_local": v_local_arr, "pos_global": np.array(pos_global_hist),
                "v_s": v_s_arr
            }
            
        ideal = runs_data["Ideal"]
        noisy = runs_data["With Disturbances"]

        fig, axs = plt.subplots(3, 2, figsize=(15, 15))
        fig.suptitle(f"Asymmetric Block (30-45 Alignment) | {m['name']}", fontsize=16, fontweight='bold')
        
        # Subplot 1: Voltage Inputs
        for w in range(4): 
            axs[0, 0].plot(t_vec, ideal["V"][:, w], label=f'V{w+1}')
        axs[0, 0].set_title("Applied Motor Terminal Voltages"); axs[0, 0].set_xlabel("Time (s)"); axs[0, 0].set_ylabel("Voltage (V)"); axs[0, 0].grid(True); axs[0, 0].legend(loc="upper right")
        
        # Subplot 2: Wheel Speeds (FIXED: Raw string notation used)
        for w in range(4):
            axs[0, 1].plot(t_vec, ideal["omega"][:, w], '-', color=f'C{w}', label=r'$\omega_{' + str(w+1) + r'}$ Ideal', **style_ideal)
            axs[0, 1].plot(t_vec, noisy["omega"][:, w], '-', color=f'C{w}', label=r'$\omega_{' + str(w+1) + r'}$ Noise', **style_noisy)
        axs[0, 1].set_title("Wheel Angular Velocities Comparison"); axs[0, 1].set_xlabel("Time (s)"); axs[0, 1].set_ylabel("Speed (rad/s)"); axs[0, 1].grid(True); axs[0, 1].legend(loc="upper right", ncol=2, fontsize='small')
        
        # Subplot 3: Local Frame Chassis Velocities (FIXED: Line parameters isolated)
        axs[1, 0].plot(t_vec, ideal["v_local"][:, 0], 'r-', label=r'$\dot{x}_r$ Ideal', **style_ideal)
        axs[1, 0].plot(t_vec, noisy["v_local"][:, 0], 'r-', label=r'$\dot{x}_r$ Noise', **style_noisy)
        axs[1, 0].plot(t_vec, ideal["v_local"][:, 1], 'b-', label=r'$\dot{y}_r$ Ideal', **style_ideal)
        axs[1, 0].plot(t_vec, noisy["v_local"][:, 1], 'b-', label=r'$\dot{y}_r$ Noise', **style_noisy)
        axs[1, 0].plot(t_vec, ideal["v_local"][:, 2], 'g-', label=r'$\dot{\theta}$ Ideal', **style_ideal)
        axs[1, 0].plot(t_vec, noisy["v_local"][:, 2], 'g-', label=r'$\dot{\theta}$ Noise', **style_noisy)
        axs[1, 0].set_title("Chassis Local Frame Speeds"); axs[1, 0].set_xlabel("Time (s)"); axs[1, 0].set_ylabel("Velocity (m/s or rad/s)"); axs[1, 0].grid(True); axs[1, 0].legend(loc="upper right", ncol=2, fontsize='small')
        
        # Subplot 4: Global Trajectory Plane
        axs[1, 1].plot(ideal["pos_global"][:, 0], ideal["pos_global"][:, 1], 'k--', lw=2.0, label='Ideal Base Path')
        axs[1, 1].plot(noisy["pos_global"][:, 0], noisy["pos_global"][:, 1], 'b-', lw=1.5, label='Perturbed Path')
        axs[1, 1].plot(noisy["pos_global"][0, 0], noisy["pos_global"][0, 1], 'go', markersize=8, label='Start')
        axs[1, 1].plot(noisy["pos_global"][-1, 0], noisy["pos_global"][-1, 1], 'rx', markersize=8, label='End')
        axs[1, 1].set_title("Global Spatial Position Tracking (X-Y)"); axs[1, 1].set_xlabel("Global X Position (m)"); axs[1, 1].set_ylabel("Global Y Position (m)"); axs[1, 1].set_aspect('equal', 'datalim'); axs[1, 1].grid(True); axs[1, 1].legend(loc="best")
        
        # Subplot 5: Wheel Contact Slip Velocity (Option 2)
        for w in range(4):
            axs[2, 0].plot(t_vec, ideal["v_s"][:, w], color=f'C{w}', label=f'$v_{{s,{w+1}}}$ Ideal', **style_ideal)
            axs[2, 0].plot(t_vec, noisy["v_s"][:, w], color=f'C{w}', label=f'$v_{{s,{w+1}}}$ Noise', **style_noisy)
        
        axs[2, 0].axhline(slip_limit, color='r', linestyle='--', alpha=0.7, label=f'Sliding Limit (±{slip_limit*1000:.1f} mm/s)')
        axs[2, 0].axhline(-slip_limit, color='r', linestyle='--', alpha=0.7)
        axs[2, 0].set_title("Wheel Contact Slip Velocity (v_s in m/s)")
        axs[2, 0].set_xlabel("Time (s)"); axs[2, 0].set_ylabel("Slip Velocity (m/s)")
        axs[2, 0].grid(True); axs[2, 0].legend(loc="upper right", ncol=2, fontsize='small')

        # Subplot 6: Wheel Active Sliding Binary Indicator (|v_s| > 5 mm/s)
        for w in range(4):
            axs[2, 1].plot(t_vec, (np.abs(ideal["v_s"][:, w]) > slip_limit).astype(float), color=f'C{w}', label=f'Wheel {w+1} Sliding')
        axs[2, 1].set_title(f"Wheel Active Sliding Status Indicator (|v_s| > {slip_limit*1000:.1f} mm/s)")
        axs[2, 1].set_xlabel("Time (s)"); axs[2, 1].set_ylabel("Sliding Flag (1=Sliding, 0=Rolling)")
        axs[2, 1].set_ylim(-0.1, 1.1)
        axs[2, 1].grid(True); axs[2, 1].legend(loc="upper right", ncol=2, fontsize='small')

        plt.tight_layout()
        plt.show()

    # Safely restore baseline parameter noise variables back to configuration dictionary
    simulation_parameters['sigma_V'] = nominal_sigma_V
    simulation_parameters['sigma_mu'] = nominal_sigma_mu


In [10]:
if run_OpenLoop_Simulation:

    # =======================================================================
    # CONFIGURATION CHANGE: KEEP ASYMMETRIC 30-45 DESIGN BUT BOOSTED
    # =======================================================================
    simulation_parameters['alpha_f'] = np.radians(30.0)
    simulation_parameters['alpha_b'] = np.radians(45.0)



    # Define high-voltage directional maneuvers matching your verified sign choices
    high_volt_maneuvers = [
        {"name": "Test 1: Forward Motion (High Volt)", "V": np.array([bost_voltage, -bost_voltage, -bost_voltage, bost_voltage])},
        {"name": "Test 2: Lateral Side-Strafing (High Volt)", "V": np.array([-bost_voltage, -bost_voltage, bost_voltage, bost_voltage])},
        {"name": "Test 3: Pure Rotational Spinning (High Volt)", "V": np.array([bost_voltage, bost_voltage, bost_voltage, bost_voltage])}
    ]

    print("Executing Block 3: Asymmetric 30-45 Wheel Angle Configurations at High Voltage (6.0V)...")

    for m in high_volt_maneuvers:
        runs_data = {}
        for condition in ["Ideal", "With Disturbances"]:
            if condition == "Ideal":
                simulation_parameters['sigma_V'] = 0.0
                simulation_parameters['sigma_mu'] = 0.0
            else:
                simulation_parameters['sigma_V'] = nominal_sigma_V
                simulation_parameters['sigma_mu'] = nominal_sigma_mu
                
            # Instantiate plant under custom asymmetric high-volt conditions (Seeded for reproducibility)
            if model_choice == 1:
                robot_plant = OmniRobotSimulationExplicit(simulation_parameters, random_seed=42)
                state = OmniRobotStateExplicit()    
            elif model_choice == 2:   
                robot_plant = OmniRobotSimulationODESolve(simulation_parameters, random_seed=42)
                state = OmniRobotStateODESolve()
            
            
            v_in_hist, omega_hist, v_local_hist, pos_global_hist = [], [], [], []
            
            for k in range(steps):
                v_in_hist.append(m["V"].copy())
                omega_hist.append(state.omega.copy())
                v_local_hist.append(state.v_local.copy())
                pos_global_hist.append(state.pos_global.copy())
                state = robot_plant.step(state, m["V"])
                
            omega_arr = np.array(omega_hist)
            v_local_arr = np.array(v_local_hist)
            v_s_arr = (v_local_arr @ robot_plant.H.T) - robot_plant.p['r'] * omega_arr
            runs_data[condition] = {
                "V": np.array(v_in_hist), "omega": omega_arr,
                "v_local": v_local_arr, "pos_global": np.array(pos_global_hist),
                "v_s": v_s_arr
            }
            
        ideal = runs_data["Ideal"]
        noisy = runs_data["With Disturbances"]

        fig, axs = plt.subplots(3, 2, figsize=(15, 15))
        fig.suptitle(f"Asymmetric High-Volt Block (30-45 @ 6.0V) | {m['name']}", fontsize=16, fontweight='bold')
        
        # Subplot 1: Voltage Inputs
        for w in range(4): 
            axs[0, 0].plot(t_vec, ideal["V"][:, w], label=f'V{w+1}')
        axs[0, 0].set_title("Applied Motor Terminal Voltages"); axs[0, 0].set_xlabel("Time (s)"); axs[0, 0].set_ylabel("Voltage (V)"); axs[0, 0].grid(True); axs[0, 0].legend(loc="upper right")
        
        # Subplot 2: Wheel Speeds
        for w in range(4):
            axs[0, 1].plot(t_vec, ideal["omega"][:, w], '-', color=f'C{w}', label=r'$\omega_{' + str(w+1) + r'}$ Ideal', **style_ideal)
            axs[0, 1].plot(t_vec, noisy["omega"][:, w], '-', color=f'C{w}', label=r'$\omega_{' + str(w+1) + r'}$ Noise', **style_noisy)
        axs[0, 1].set_title("Wheel Angular Velocities Comparison"); axs[0, 1].set_xlabel("Time (s)"); axs[0, 1].set_ylabel("Speed (rad/s)"); axs[0, 1].grid(True); axs[0, 1].legend(loc="upper right", ncol=2, fontsize='small')
        
        # Subplot 3: Local Frame Chassis Velocities
        axs[1, 0].plot(t_vec, ideal["v_local"][:, 0], 'r-', label=r'$\dot{x}_r$ Ideal', **style_ideal)
        axs[1, 0].plot(t_vec, noisy["v_local"][:, 0], 'r-', label=r'$\dot{x}_r$ Noise', **style_noisy)
        axs[1, 0].plot(t_vec, ideal["v_local"][:, 1], 'b-', label=r'$\dot{y}_r$ Ideal', **style_ideal)
        axs[1, 0].plot(t_vec, noisy["v_local"][:, 1], 'b-', label=r'$\dot{y}_r$ Noise', **style_noisy)
        axs[1, 0].plot(t_vec, ideal["v_local"][:, 2], 'g-', label=r'$\dot{\theta}$ Ideal', **style_ideal)
        axs[1, 0].plot(t_vec, noisy["v_local"][:, 2], 'g-', label=r'$\dot{\theta}$ Noise', **style_noisy)
        axs[1, 0].set_title("Chassis Local Frame Speeds"); axs[1, 0].set_xlabel("Time (s)"); axs[1, 0].set_ylabel("Velocity (m/s or rad/s)"); axs[1, 0].grid(True); axs[1, 0].legend(loc="upper right", ncol=2, fontsize='small')
        
        # Subplot 4: Global Trajectory Plane
        axs[1, 1].plot(ideal["pos_global"][:, 0], ideal["pos_global"][:, 1], 'k--', lw=2.0, label='Ideal Base Path')
        axs[1, 1].plot(noisy["pos_global"][:, 0], noisy["pos_global"][:, 1], 'b-', lw=1.5, label='Perturbed Path')
        axs[1, 1].plot(noisy["pos_global"][0, 0], noisy["pos_global"][0, 1], 'go', markersize=8, label='Start')
        axs[1, 1].plot(noisy["pos_global"][-1, 0], noisy["pos_global"][-1, 1], 'rx', markersize=8, label='End')
        axs[1, 1].set_title("Global Spatial Position Tracking (X-Y)"); axs[1, 1].set_xlabel("Global X Position (m)"); axs[1, 1].set_ylabel("Global Y Position (m)"); axs[1, 1].set_aspect('equal', 'datalim'); axs[1, 1].grid(True); axs[1, 1].legend(loc="best")
        
        # Subplot 5: Wheel Contact Slip Velocity (Option 2)
        for w in range(4):
            axs[2, 0].plot(t_vec, ideal["v_s"][:, w], color=f'C{w}', label=f'$v_{{s,{w+1}}}$ Ideal', **style_ideal)
            axs[2, 0].plot(t_vec, noisy["v_s"][:, w], color=f'C{w}', label=f'$v_{{s,{w+1}}}$ Noise', **style_noisy)
        axs[2, 0].axhline(slip_limit, color='r', linestyle='--', alpha=0.7, label=f'Sliding Limit (±{slip_limit*1000:.1f} mm/s)')
        axs[2, 0].axhline(-slip_limit, color='r', linestyle='--', alpha=0.7)
        axs[2, 0].set_title("Wheel Contact Slip Velocity (v_s in m/s)")
        axs[2, 0].set_xlabel("Time (s)"); axs[2, 0].set_ylabel("Slip Velocity (m/s)")
        axs[2, 0].grid(True); axs[2, 0].legend(loc="upper right", ncol=2, fontsize='small')

        # Subplot 6: Wheel Active Sliding Binary Indicator (|v_s| > 5 mm/s)
        for w in range(4):
            axs[2, 1].plot(t_vec, (np.abs(ideal["v_s"][:, w]) > slip_limit).astype(float), color=f'C{w}', label=f'Wheel {w+1} Sliding')
        axs[2, 1].set_title(f"Wheel Active Sliding Status Indicator (|v_s| > {slip_limit*1000:.1f} mm/s)")
        axs[2, 1].set_xlabel("Time (s)"); axs[2, 1].set_ylabel("Sliding Flag (1=Sliding, 0=Rolling)")
        axs[2, 1].set_ylim(-0.1, 1.1)
        axs[2, 1].grid(True); axs[2, 1].legend(loc="upper right", ncol=2, fontsize='small')

        plt.tight_layout()
        plt.show()

    # Final safe restoration of baseline parameter noise variables back to dictionary
    simulation_parameters['sigma_V'] = nominal_sigma_V
    simulation_parameters['sigma_mu'] = nominal_sigma_mu
